# Threads Reader Bot — полный установщик Cloudflare

Этот ноутбук **сам создаёт текущий TypeScript-код** через `%%writefile`, создаёт новый GitHub-репозиторий, D1, Queue, импортирует приватные cookies и развёртывает Worker.

### Безопасность
- GitHub PAT, Cloudflare API Token и токены бота вводятся скрыто через `getpass`.
- Ни один токен не записывается в создаваемые файлы.
- Cookies находятся только во временной директории Colab, импортируются прямо в D1 и удаляются в `finally`.
- Cookies не попадают в Git-коммит.
- Worker автоматически обновляет cookies аккаунтов в D1 после успешных запросов Threads.
- После завершения обязательно выполните последнюю ячейку и выберите **Runtime → Disconnect and delete runtime**.


## 1. Подготовка директорий


In [ ]:
from pathlib import Path
import os, shutil, subprocess, json, re, getpass, secrets, requests, base64, gc

PROJECT = Path("/content/threadsbot-cloudflare")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
for folder in ("src", "migrations", "scripts", "test", "notebooks"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print("✅ Пустой проект подготовлен:", PROJECT)


## 2. Создание файлов через `%%writefile`

Ниже 17 ячеек. Это точная копия текущего Cloudflare-кода; запускайте их по порядку.


### `.gitignore`


In [ ]:
%%writefile /content/threadsbot-cloudflare/.gitignore
# Секреты
.env
.env.local
accounts/
*.json

# Python
__pycache__/
*.pyc
venv/
*.log

# Архивы и данные
*.7z
*.zip
sample_data/
.config/

# TypeScript / Cloudflare generated files
node_modules/
.wrangler/
.accounts-import.sql
.legacy-import.sql

# TypeScript/Cloudflare project metadata
!package.json
!tsconfig.json
!package-lock.json


### `README.md`


In [ ]:
%%writefile /content/threadsbot-cloudflare/README.md
# Threads Reader Bot — Cloudflare Workers

TypeScript-версия Telegram-бота для Cloudflare Workers. Функционал Python-версии сохранён:

- чтение постов Threads как текстом, так и скриншотами;
- загрузка комментариев ответом на сообщение с постом;
- ротация нескольких «технических» Threads-аккаунтов по cookies;
- бесплатные дневные/месячные лимиты, rate limit и кеш;
- Telegram Stars и Crypto Bot;
- RU/EN/DE/ES/PT, поддержка, тикеты, баны и админская аналитика;
- ежедневный отчёт администраторам.

## Архитектура

- **Worker webhook** вместо постоянно работающего Python polling-процесса;
- **Cloudflare D1** вместо локального SQLite и оперативных словарей;
- **Cloudflare Browser Rendering** (`@cloudflare/playwright`) вместо локального Chromium;
- cookies аккаунтов хранятся в D1, автоматически обновляются после успешного запроса.

> Browser Rendering должен быть включён в аккаунте Cloudflare. Его стоимость и лимиты зависят от тарифа Cloudflare — сам Worker не может запускать обычный серверный Chromium.

## Развёртывание

### Google Colab — установка по ячейкам

Откройте [`notebooks/threadsbot_cloudflare_deploy.ipynb`](notebooks/threadsbot_cloudflare_deploy.ipynb) в Google Colab. Это самостоятельный установщик: исходники создаются прямо в Colab отдельными `%%writefile`-ячейками, поэтому предварительно клонировать репозиторий не требуется. Ноутбук:

1. создаёт все файлы TypeScript-проекта из текущей версии кода;
2. скрыто принимает GitHub Personal Access Token, создаёт репозиторий и пушит код;
3. скрыто принимает Cloudflare API Token и Account ID;
4. создаёт или повторно использует D1 и Queue, подставляя D1 UUID в `wrangler.toml`;
5. загружает cookies только во временное приватное хранилище Colab;
6. импортирует cookies непосредственно в D1 и сразу удаляет локальные файлы;
7. при необходимости переносит старый `bot.db`;
8. добавляет секреты, развёртывает Worker и устанавливает webhook.

GitHub и Cloudflare токены вводятся через `getpass`, не записываются в проект и очищаются в финальной ячейке. Cookies не отправляются в GitHub. После импорта Worker автоматически обновляет их прямо в D1 после успешных запросов Threads.

### 1. Установить зависимости

```bash
npm install
npx wrangler login
```

### 2. Создать D1 и очередь обновлений

Очередь нужна, чтобы Telegram сразу получил `200 OK`, а чтение Threads могло безопасно продолжаться дольше 30 секунд без повторных webhook-запросов.

```bash
npx wrangler d1 create threadsbot
npx wrangler queues create threadsbot-updates
```

Скопируйте полученный `database_id` в `wrangler.toml` вместо `REPLACE_WITH_D1_DATABASE_ID`, затем:

```bash
npm run db:remote
```

### 3. Добавить секреты

```bash
npx wrangler secret put TELEGRAM_TOKEN
npx wrangler secret put CRYPTO_BOT_TOKEN
npx wrangler secret put WEBHOOK_SECRET
```

`WEBHOOK_SECRET` — произвольная длинная случайная строка без пробелов. ID администраторов меняются в `wrangler.toml`.

### 4. Импортировать аккаунты Threads

Файлы остаются в прежнем формате: `accounts/name.json` (Cookie-Editor JSON или Playwright cookies). Они игнорируются Git и не попадут в репозиторий.

```bash
npm run accounts:import -- accounts --remote
```

Скрипт проверяет JSON, импортирует каждый файл в D1 и удаляет временный SQL-файл. Фейковые/технические аккаунты не убраны: бот выбирает живой аккаунт с наименьшей часовой нагрузкой, сохраняет обновлённые cookies и переключается на следующий при истёкшей сессии или ошибке.

Если нужно перенести также пользователей, подписки, лимиты, кеш, тикеты и аналитику из старого `data/bot.db`, вместо отдельного импорта выполните:

```bash
python3 scripts/migrate_legacy.py --db data/bot.db --accounts accounts
```

Скрипт переносит SQLite-таблицы и cookies в D1, не добавляя временный экспорт в Git.

### 5. Deploy и webhook

```bash
npm run deploy
curl -X POST "https://YOUR-WORKER.workers.dev/setup-webhook" \
  -H "Authorization: Bearer YOUR_WEBHOOK_SECRET"
```

Проверка состояния:

```bash
curl https://YOUR-WORKER.workers.dev/health
```

## Локальная проверка

```bash
npm run db:local
npm run typecheck
npm test
npm run dev
```

Локальная эмуляция Browser Rendering может отличаться от production. Полную проверку чтения Threads лучше делать после deploy с одним тестовым аккаунтом.

## Безопасность

- Не коммитьте `.dev.vars`, `.env`, `accounts/` и экспорт cookies.
- Cookies дают доступ к Threads-аккаунтам: используйте отдельные аккаунты с минимальными правами.
- Endpoint Telegram защищён и секретным URL, и заголовком `X-Telegram-Bot-Api-Secret-Token`.
- Пользовательский текст экранируется перед отправкой в Telegram HTML.

## Старый Python-код

`bot.py` и `threads_check.py` оставлены как референс для сверки поведения. Production entrypoint теперь `src/index.ts`.


### `package.json`


In [ ]:
%%writefile /content/threadsbot-cloudflare/package.json
{
  "name": "threads-reader-worker",
  "private": true,
  "version": "2.0.0",
  "type": "module",
  "scripts": {
    "dev": "wrangler dev",
    "deploy": "wrangler deploy",
    "typecheck": "tsc --noEmit",
    "test": "vitest run",
    "db:local": "wrangler d1 migrations apply threadsbot --local",
    "db:remote": "wrangler d1 migrations apply threadsbot --remote",
    "accounts:import": "tsx scripts/import-accounts.ts"
  },
  "dependencies": {
    "@cloudflare/playwright": "^1.1.0"
  },
  "devDependencies": {
    "@cloudflare/workers-types": "^5.20260815.1",
    "tsx": "^4.20.0",
    "typescript": "^5.9.0",
    "vitest": "^3.2.0",
    "wrangler": "^4.30.0"
  }
}


### `tsconfig.json`


In [ ]:
%%writefile /content/threadsbot-cloudflare/tsconfig.json
{
  "compilerOptions": {
    "target": "ES2022",
    "module": "ESNext",
    "moduleResolution": "Bundler",
    "lib": ["ES2022", "DOM", "DOM.Iterable"],
    "types": ["@cloudflare/workers-types"],
    "strict": true,
    "noEmit": true,
    "skipLibCheck": true,
    "resolveJsonModule": true
  },
  "include": ["src/**/*.ts", "test/**/*.ts"]
}


### `wrangler.toml`


In [ ]:
%%writefile /content/threadsbot-cloudflare/wrangler.toml
name = "threads-reader-bot"
main = "src/index.ts"
compatibility_date = "2026-08-15"
compatibility_flags = ["nodejs_compat"]

[browser]
binding = "BROWSER"

[[d1_databases]]
binding = "DB"
database_name = "threadsbot"
database_id = "REPLACE_WITH_D1_DATABASE_ID"
migrations_dir = "migrations"

[[queues.producers]]
binding = "UPDATES"
queue = "threadsbot-updates"

[[queues.consumers]]
queue = "threadsbot-updates"
max_batch_size = 1
max_batch_timeout = 1
max_retries = 3

[triggers]
crons = ["0 6 * * *", "0 * * * *"] # 09:00 Minsk report; hourly cleanup

[vars]
ADMIN_IDS = "369330135,657708753"
STATS_EXCLUDE_IDS = "369330135,657708753"
BASE_URL = "https://www.threads.com"


### `migrations/0001_initial.sql`


In [ ]:
%%writefile /content/threadsbot-cloudflare/migrations/0001_initial.sql
CREATE TABLE IF NOT EXISTS banned_users (user_id INTEGER PRIMARY KEY, username TEXT, reason TEXT, banned_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS request_log (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER NOT NULL, username_requested TEXT NOT NULL, timestamp TEXT NOT NULL);
CREATE INDEX IF NOT EXISTS idx_request_log_user_time ON request_log(user_id, timestamp);
CREATE TABLE IF NOT EXISTS cache (username TEXT, mode TEXT, page INTEGER, data TEXT NOT NULL, cached_at TEXT NOT NULL, PRIMARY KEY(username,mode,page));
CREATE TABLE IF NOT EXISTS subscriptions (user_id INTEGER PRIMARY KEY, expires_at TEXT NOT NULL, payment_method TEXT, total_paid REAL DEFAULT 0, payments_count INTEGER DEFAULT 0);
CREATE TABLE IF NOT EXISTS payments_log (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER NOT NULL, amount TEXT, method TEXT, timestamp TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS support_tickets (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER NOT NULL, username TEXT, message TEXT NOT NULL, ticket_type TEXT DEFAULT 'question', status TEXT DEFAULT 'open', created_at TEXT NOT NULL, answered_at TEXT, answer TEXT);
CREATE TABLE IF NOT EXISTS user_events (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER NOT NULL, event_type TEXT NOT NULL, event_data TEXT, timestamp TEXT NOT NULL);
CREATE INDEX IF NOT EXISTS idx_events_user_type_time ON user_events(user_id,event_type,timestamp);
CREATE INDEX IF NOT EXISTS idx_events_type_time ON user_events(event_type,timestamp);
CREATE TABLE IF NOT EXISTS user_settings (user_id INTEGER PRIMARY KEY, language TEXT DEFAULT 'ru', created_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS bot_state (scope TEXT NOT NULL, state_key TEXT NOT NULL, value TEXT NOT NULL, updated_at TEXT NOT NULL, PRIMARY KEY(scope,state_key));
CREATE TABLE IF NOT EXISTS processed_updates (update_id INTEGER PRIMARY KEY, status TEXT NOT NULL, updated_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS threads_accounts (
  name TEXT PRIMARY KEY, cookies TEXT NOT NULL, enabled INTEGER NOT NULL DEFAULT 1,
  is_alive INTEGER NOT NULL DEFAULT 1, last_error TEXT, requests_count INTEGER NOT NULL DEFAULT 0,
  posts_sent INTEGER NOT NULL DEFAULT 0, errors_count INTEGER NOT NULL DEFAULT 0,
  hourly_requests INTEGER NOT NULL DEFAULT 0, hourly_reset TEXT NOT NULL, last_used TEXT, updated_at TEXT NOT NULL
);


### `scripts/import-accounts.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/scripts/import-accounts.ts
import { readdir, readFile, rm, writeFile } from "node:fs/promises";
import { join, parse } from "node:path";
import { spawnSync } from "node:child_process";

const dir=process.argv[2]||"accounts";
const remote=process.argv.includes("--remote");
const quote=(v:string)=>`'${v.replaceAll("'","''")}'`;
const files=(await readdir(dir)).filter(x=>x.endsWith(".json"));
if(!files.length)throw new Error(`No account cookie files in ${dir}`);
const now=new Date().toISOString();
let sql="BEGIN;\n";
for(const file of files){const raw=await readFile(join(dir,file),"utf8");JSON.parse(raw);const name=parse(file).name;sql+=`INSERT INTO threads_accounts(name,cookies,enabled,is_alive,hourly_reset,updated_at) VALUES(${quote(name)},${quote(raw)},1,1,${quote(now)},${quote(now)}) ON CONFLICT(name) DO UPDATE SET cookies=excluded.cookies,enabled=1,is_alive=1,last_error=NULL,updated_at=excluded.updated_at;\n`}
sql+="COMMIT;\n";
const temp=".accounts-import.sql";await writeFile(temp,sql,{mode:0o600});
try{const args=["wrangler","d1","execute","threadsbot",remote?"--remote":"--local","--file",temp];const result=spawnSync("npx",args,{stdio:"inherit",shell:process.platform==="win32"});if(result.status!==0)process.exit(result.status||1)}finally{await rm(temp,{force:true})}
console.log(`Imported ${files.length} Threads account(s).`);


### `scripts/migrate_legacy.py`


In [ ]:
%%writefile /content/threadsbot-cloudflare/scripts/migrate_legacy.py
#!/usr/bin/env python3
"""Import the Python bot's SQLite data and account cookies into Cloudflare D1."""
import argparse, json, sqlite3, subprocess
from pathlib import Path

TABLES = ["banned_users", "request_log", "cache", "subscriptions", "payments_log", "support_tickets", "user_events", "user_settings"]

def quote(value):
    if value is None: return "NULL"
    if isinstance(value, (int, float)): return str(value)
    return "'" + str(value).replace("'", "''") + "'"

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--db", default="data/bot.db")
    p.add_argument("--accounts", default="accounts")
    p.add_argument("--local", action="store_true")
    args=p.parse_args()
    conn=sqlite3.connect(args.db); conn.row_factory=sqlite3.Row
    lines=["BEGIN;"]
    existing={r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    for table in TABLES:
        if table not in existing: continue
        for row in conn.execute(f'SELECT * FROM "{table}"'):
            columns=",".join(f'"{k}"' for k in row.keys())
            values=",".join(quote(row[k]) for k in row.keys())
            lines.append(f'INSERT OR REPLACE INTO "{table}"({columns}) VALUES({values});')
    now=__import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat()
    account_dir=Path(args.accounts)
    if account_dir.exists():
        for file in sorted(account_dir.glob("*.json")):
            raw=file.read_text("utf-8"); json.loads(raw)
            lines.append("INSERT INTO threads_accounts(name,cookies,enabled,is_alive,hourly_reset,updated_at) "
                         f"VALUES({quote(file.stem)},{quote(raw)},1,1,{quote(now)},{quote(now)}) "
                         "ON CONFLICT(name) DO UPDATE SET cookies=excluded.cookies,enabled=1,is_alive=1,updated_at=excluded.updated_at;")
    lines.append("COMMIT;")
    temp=Path(".legacy-import.sql")
    try:
        temp.write_text("\n".join(lines),"utf-8")
        command=["npx","wrangler","d1","execute","threadsbot","--local" if args.local else "--remote","--file",str(temp)]
        subprocess.run(command,check=True)
    finally: temp.unlink(missing_ok=True)

if __name__ == "__main__": main()


### `src/config.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/config.ts
export interface Env {
  DB: D1Database;
  BROWSER: import("@cloudflare/playwright").BrowserWorker;
  UPDATES: Queue<import("./telegram").TelegramUpdate>;
  TELEGRAM_TOKEN: string;
  CRYPTO_BOT_TOKEN: string;
  WEBHOOK_SECRET: string;
  ADMIN_IDS?: string;
  STATS_EXCLUDE_IDS?: string;
  BASE_URL?: string;
}

export const LIMITS = {
  priceStars: 150,
  priceCryptoUsd: 2.5,
  subscriptionDays: 30,
  freeMonthly: 10,
  freeDaily: 3,
  perMinute: 3,
  perHour: 15,
  perDay: 50,
  cacheMinutes: 5,
  accountHourly: 20,
} as const;

export const adminIds = (env: Env): number[] =>
  (env.ADMIN_IDS || "369330135,657708753").split(",").map(Number).filter(Number.isFinite);
export const excludedIds = (env: Env): number[] =>
  (env.STATS_EXCLUDE_IDS || env.ADMIN_IDS || "").split(",").map(Number).filter(Number.isFinite);
export const isAdmin = (env: Env, uid: number): boolean => adminIds(env).includes(uid);


### `src/db.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/db.ts
import { LIMITS, type Env, excludedIds } from "./config";

const now = () => new Date().toISOString();
const since = (ms: number) => new Date(Date.now() - ms).toISOString();
export type StateName = "last_button" | "last_username" | "waiting_support" | "admin_reply";

export class Database {
  constructor(private readonly env: Env) {}
  private get db() { return this.env.DB; }

  async getLang(uid: number): Promise<string> {
    return (await this.db.prepare("SELECT language FROM user_settings WHERE user_id=?").bind(uid).first<{language:string}>())?.language || "ru";
  }
  async hasLang(uid: number): Promise<boolean> { return !!await this.db.prepare("SELECT 1 x FROM user_settings WHERE user_id=?").bind(uid).first(); }
  setLang(uid: number, lang: string) { return this.db.prepare("INSERT INTO user_settings VALUES(?,?,?) ON CONFLICT(user_id) DO UPDATE SET language=excluded.language").bind(uid, lang, now()).run(); }
  isBanned(uid: number) { return this.db.prepare("SELECT 1 x FROM banned_users WHERE user_id=?").bind(uid).first().then(Boolean); }
  ban(uid: number, reason: string) { return this.db.prepare("INSERT INTO banned_users(user_id,reason,banned_at) VALUES(?,?,?) ON CONFLICT(user_id) DO UPDATE SET reason=excluded.reason,banned_at=excluded.banned_at").bind(uid, reason, now()).run(); }
  unban(uid: number) { return this.db.prepare("DELETE FROM banned_users WHERE user_id=?").bind(uid).run(); }
  async banned() { return (await this.db.prepare("SELECT * FROM banned_users ORDER BY banned_at DESC").all()).results; }

  logEvent(uid: number, type: string, data = "") { return this.db.prepare("INSERT INTO user_events(user_id,event_type,event_data,timestamp) VALUES(?,?,?,?)").bind(uid,type,data,now()).run(); }
  logRequest(uid: number, username: string) { return this.db.prepare("INSERT INTO request_log(user_id,username_requested,timestamp) VALUES(?,?,?)").bind(uid,username,now()).run(); }
  async usage(uid: number): Promise<{daily:number;monthly:number}> {
    const d = new Date(); d.setUTCHours(0,0,0,0);
    const m = new Date(Date.UTC(d.getUTCFullYear(), d.getUTCMonth(), 1));
    const row = await this.db.prepare("SELECT SUM(timestamp>=?) daily, COUNT(*) monthly FROM user_events WHERE user_id=? AND event_type='free_request' AND timestamp>=?").bind(d.toISOString(),uid,m.toISOString()).first<{daily:number;monthly:number}>();
    return { daily: Number(row?.daily || 0), monthly: Number(row?.monthly || 0) };
  }
  async rateLimit(uid: number): Promise<string | null> {
    const row = await this.db.prepare("SELECT SUM(timestamp>?) m, SUM(timestamp>?) h, COUNT(*) d FROM request_log WHERE user_id=? AND timestamp>?").bind(since(60_000),since(3_600_000),uid,since(86_400_000)).first<{m:number;h:number;d:number}>();
    if (Number(row?.m||0) >= LIMITS.perMinute) return `Лимит ${LIMITS.perMinute}/мин.`;
    if (Number(row?.h||0) >= LIMITS.perHour) return `Лимит ${LIMITS.perHour}/час.`;
    if (Number(row?.d||0) >= LIMITS.perDay) return `Лимит ${LIMITS.perDay}/сутки.`;
    return null;
  }
  async subscription(uid: number): Promise<(Record<string, unknown> & { expires_at: string; active: boolean; days_left: number }) | null> {
    const row = await this.db.prepare("SELECT * FROM subscriptions WHERE user_id=?").bind(uid).first<Record<string,unknown>>();
    if (!row) return null;
    const expires_at = String(row.expires_at);
    const delta = new Date(expires_at).getTime() - Date.now();
    return { ...row, expires_at, active: delta > 0, days_left: Math.max(0, Math.floor(delta/86_400_000)) };
  }
  async hasSubscription(uid: number) { return (await this.subscription(uid))?.active === true; }
  async activate(uid: number, method: string, amount: number): Promise<Date> {
    const old = await this.subscription(uid);
    const base = old?.active ? new Date(String(old.expires_at)) : new Date();
    const expiry = new Date(base.getTime() + LIMITS.subscriptionDays*86_400_000);
    await this.db.batch([
      this.db.prepare("INSERT INTO subscriptions(user_id,expires_at,payment_method,total_paid,payments_count) VALUES(?,?,?,?,1) ON CONFLICT(user_id) DO UPDATE SET expires_at=excluded.expires_at,payment_method=excluded.payment_method,total_paid=total_paid+excluded.total_paid,payments_count=payments_count+1").bind(uid,expiry.toISOString(),method,amount),
      this.db.prepare("INSERT INTO payments_log(user_id,amount,method,timestamp) VALUES(?,?,?,?)").bind(uid,String(amount),method,now()),
    ]);
    return expiry;
  }
  async subscribers() { return (await this.db.prepare("SELECT * FROM subscriptions WHERE expires_at>?").bind(now()).all()).results; }

  async cache<T>(username:string, mode:string, page=0): Promise<T|null> {
    const row = await this.db.prepare("SELECT data,cached_at FROM cache WHERE username=? AND mode=? AND page=?").bind(username,mode,page).first<{data:string;cached_at:string}>();
    if (!row || Date.now()-new Date(row.cached_at).getTime() >= LIMITS.cacheMinutes*60_000) return null;
    return JSON.parse(row.data) as T;
  }
  setCache(username:string, mode:string, data:unknown, page=0) { return this.db.prepare("INSERT INTO cache VALUES(?,?,?,?,?) ON CONFLICT(username,mode,page) DO UPDATE SET data=excluded.data,cached_at=excluded.cached_at").bind(username,mode,page,JSON.stringify(data),now()).run(); }

  async state(scope:string|number,key:StateName): Promise<string|null> { return (await this.db.prepare("SELECT value FROM bot_state WHERE scope=? AND state_key=?").bind(String(scope),key).first<{value:string}>())?.value || null; }
  setState(scope:string|number,key:StateName,value:string) { return this.db.prepare("INSERT INTO bot_state VALUES(?,?,?,?) ON CONFLICT(scope,state_key) DO UPDATE SET value=excluded.value,updated_at=excluded.updated_at").bind(String(scope),key,value,now()).run(); }
  clearState(scope:string|number,key:StateName) { return this.db.prepare("DELETE FROM bot_state WHERE scope=? AND state_key=?").bind(String(scope),key).run(); }

  async createTicket(uid:number, username:string, message:string, type:string): Promise<number> { const r=await this.db.prepare("INSERT INTO support_tickets(user_id,username,message,ticket_type,status,created_at) VALUES(?,?,?,?,'open',?)").bind(uid,username,message,type,now()).run(); return Number(r.meta.last_row_id); }
  ticket(id:number) { return this.db.prepare("SELECT * FROM support_tickets WHERE id=?").bind(id).first<Record<string,unknown>>(); }
  async tickets(uid?:number) { const q=uid?this.db.prepare("SELECT * FROM support_tickets WHERE user_id=? ORDER BY created_at DESC LIMIT 5").bind(uid):this.db.prepare("SELECT * FROM support_tickets WHERE status='open' ORDER BY created_at DESC"); return (await q.all()).results; }
  answerTicket(id:number, answer:string) { return this.db.prepare("UPDATE support_tickets SET status='answered',answer=?,answered_at=? WHERE id=?").bind(answer,now(),id).run(); }

  async accountCounts() { return await this.db.prepare("SELECT COUNT(*) total,SUM(enabled) enabled,SUM(enabled AND is_alive) alive FROM threads_accounts").first<{total:number;enabled:number;alive:number}>() || {total:0,enabled:0,alive:0}; }
  async accountStats() { return (await this.db.prepare("SELECT name,is_alive,last_error,requests_count,posts_sent,errors_count,hourly_requests,hourly_reset,last_used FROM threads_accounts ORDER BY name").all()).results; }

  async analytics() {
    const excluded=excludedIds(this.env); const marks=excluded.map(()=>"?").join(","); const clause=excluded.length?` AND user_id NOT IN (${marks})`:"";
    const one=since(86_400_000), seven=since(7*86_400_000);
    const queries = [
      this.db.prepare(`SELECT COUNT(DISTINCT user_id) c FROM user_events WHERE event_type='start' AND timestamp>?${clause}`).bind(one,...excluded),
      this.db.prepare(`SELECT COUNT(DISTINCT user_id) c FROM user_events WHERE timestamp>?${clause}`).bind(one,...excluded),
      this.db.prepare(`SELECT COUNT(DISTINCT user_id) c FROM user_events WHERE timestamp>?${clause}`).bind(seven,...excluded),
      this.db.prepare(`SELECT COUNT(*) c FROM user_events WHERE event_type IN ('request','search') AND timestamp>?${clause}`).bind(one,...excluded),
      this.db.prepare(`SELECT COUNT(*) c FROM user_events WHERE event_type='free_exhausted' AND timestamp>?${clause}`).bind(one,...excluded),
      this.db.prepare(`SELECT event_data,COUNT(*) c FROM user_events WHERE event_type='request' AND timestamp>?${clause} GROUP BY event_data`).bind(one,...excluded),
      this.db.prepare(`SELECT COUNT(*) c FROM user_events WHERE event_type='subscribe' AND timestamp>?${clause}`).bind(one,...excluded),
      this.db.prepare(`SELECT COALESCE(SUM(total_paid),0) c FROM subscriptions WHERE expires_at>?${clause}`).bind(since(30*86_400_000),...excluded),
    ];
    const r=await this.db.batch(queries); const modes=r[5].results as {event_data:string;c:number}[];
    const count=(i:number)=>Number((r[i].results[0] as {c:number}|undefined)?.c||0);
    return {newUsers:count(0),dau:count(1),active7d:count(2),requests:count(3),exhausted:count(4),newSubs:count(6),revenue:count(7),text:modes.filter(x=>x.event_data.startsWith('text:')).reduce((a,x)=>a+Number(x.c),0),img:modes.filter(x=>x.event_data.startsWith('img:')).reduce((a,x)=>a+Number(x.c),0),comments:modes.filter(x=>x.event_data.startsWith('comments:')).reduce((a,x)=>a+Number(x.c),0)};
  }
  cleanup() { return this.db.batch([this.db.prepare("DELETE FROM request_log WHERE timestamp<?").bind(since(2*86_400_000)),this.db.prepare("DELETE FROM cache WHERE cached_at<?").bind(since(LIMITS.cacheMinutes*60_000)),this.db.prepare("DELETE FROM bot_state WHERE updated_at<? AND state_key IN ('waiting_support','admin_reply')").bind(since(7*86_400_000)),this.db.prepare("DELETE FROM processed_updates WHERE status='done' AND updated_at<?").bind(since(7*86_400_000))]); }
}


### `src/i18n.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/i18n.ts
import { LANGUAGE_NAMES, TRANSLATIONS } from "./translations";

export type Lang = keyof typeof TRANSLATIONS;
export { LANGUAGE_NAMES };
export const languages = Object.keys(TRANSLATIONS) as Lang[];

export function text(key: string, lang: string = "ru", values: Record<string, string | number> = {}): string {
  const selected = languages.includes(lang as Lang) ? lang as Lang : "ru";
  const table = TRANSLATIONS[selected] as Record<string, string>;
  const fallback = TRANSLATIONS.en as Record<string, string>;
  let value = table[key] ?? fallback[key] ?? (TRANSLATIONS.ru as Record<string, string>)[key] ?? key;
  for (const [name, replacement] of Object.entries(values)) {
    value = value.replaceAll(`{${name}}`, String(replacement));
  }
  return value;
}

const STRIP_WORDS = ["Translate", "Перевести", "See translation", "See more", "Показать перевод"];
export function cleanPostText(input: string): string {
  let value = input.trim();
  for (const word of STRIP_WORDS) {
    if (value.endsWith(`\n${word}`)) value = value.slice(0, -(word.length + 1)).trim();
    else if (value.endsWith(`  ${word}`)) value = value.slice(0, -(word.length + 2)).trim();
    else if (value.endsWith(word)) value = value.slice(0, -word.length).trim();
  }
  return value;
}


### `src/index.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/index.ts
import { Bot } from "./bot";
import { adminIds, type Env } from "./config";
import { Database } from "./db";
import { Telegram, type TelegramUpdate } from "./telegram";
import { reviveAccounts } from "./threads";

async function dailyReport(env:Env){const db=new Database(env),tg=new Telegram(env.TELEGRAM_TOKEN),a=await db.analytics(),accounts=await db.accountCounts(),tickets=await db.tickets();const date=new Intl.DateTimeFormat("ru-RU",{timeZone:"Europe/Minsk"}).format(new Date());const value=`📊 <b>Daily Report</b> (${date})\n\n👥 <b>Users:</b>\n   New today: ${a.newUsers}\n   New subs: ${a.newSubs}\n\n💰 <b>Revenue (approx):</b> $${a.revenue.toFixed(2)}\n\n🤖 <b>Accounts:</b> ${accounts.alive||0}/${accounts.total} alive\n\n🆘 <b>Open tickets:</b> ${tickets.length}`;await Promise.all(adminIds(env).map(id=>tg.sendMessage(id,value).catch(()=>{})))}

export default {
 async fetch(request:Request,env:Env,ctx:ExecutionContext):Promise<Response>{
  const url=new URL(request.url);
  if(url.pathname==="/health") { const accounts=await new Database(env).accountCounts();return Response.json({ok:true,accounts}); }
  if(url.pathname==="/setup-webhook"&&request.method==="POST"){
   if(request.headers.get("authorization")!==`Bearer ${env.WEBHOOK_SECRET}`)return new Response("Unauthorized",{status:401});
   const webhook=`${url.origin}/telegram/${env.WEBHOOK_SECRET}`;
   const response=await fetch(`https://api.telegram.org/bot${env.TELEGRAM_TOKEN}/setWebhook`,{method:"POST",headers:{"content-type":"application/json"},body:JSON.stringify({url:webhook,secret_token:env.WEBHOOK_SECRET,allowed_updates:["message","callback_query","pre_checkout_query"],drop_pending_updates:false})});
   return new Response(response.body,{status:response.status,headers:{"content-type":"application/json"}});
  }
  if(url.pathname!==`/telegram/${env.WEBHOOK_SECRET}`||request.method!=="POST")return new Response("Not found",{status:404});
  if(request.headers.get("x-telegram-bot-api-secret-token")!==env.WEBHOOK_SECRET)return new Response("Forbidden",{status:403});
  const update=await request.json<TelegramUpdate>();
  await env.UPDATES.send(update);
  return new Response("OK");
 },
 async queue(batch:MessageBatch<TelegramUpdate>,env:Env){
  for(const message of batch.messages){const update=message.body;const existing=await env.DB.prepare("SELECT status FROM processed_updates WHERE update_id=?").bind(update.update_id).first<{status:string}>();if(existing?.status==="done"){message.ack();continue}await env.DB.prepare("INSERT INTO processed_updates VALUES(?,'processing',?) ON CONFLICT(update_id) DO UPDATE SET status='processing',updated_at=excluded.updated_at").bind(update.update_id,new Date().toISOString()).run();try{await new Bot(env).update(update);await env.DB.prepare("UPDATE processed_updates SET status='done',updated_at=? WHERE update_id=?").bind(new Date().toISOString(),update.update_id).run();message.ack()}catch(error){console.error(error);await env.DB.prepare("DELETE FROM processed_updates WHERE update_id=?").bind(update.update_id).run();const tg=new Telegram(env.TELEGRAM_TOKEN);await Promise.all(adminIds(env).map(id=>tg.sendMessage(id,`🚨 <b>ALERT</b>\n\nBot error: ${String(error).slice(0,500)}`).catch(()=>{})));message.retry()}}
 },
 async scheduled(controller:ScheduledController,env:Env,ctx:ExecutionContext){if(controller.cron==="0 6 * * *")ctx.waitUntil(dailyReport(env));else ctx.waitUntil(Promise.all([new Database(env).cleanup(),reviveAccounts(env)]).then(()=>{}));}
} satisfies ExportedHandler<Env, TelegramUpdate>;


### `src/telegram.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/telegram.ts
export type InlineButton = { text: string; callback_data?: string; url?: string };
export type Keyboard = { inline_keyboard: InlineButton[][] };

export interface TgUser { id: number; is_bot?: boolean; username?: string; language_code?: string; }
export interface TgMessage {
  message_id: number; chat: { id: number }; from?: TgUser; text?: string; caption?: string;
  reply_to_message?: TgMessage;
  successful_payment?: { total_amount: number; invoice_payload: string };
}
export interface CallbackQuery { id: string; from: TgUser; data?: string; message?: TgMessage; }
export interface TelegramUpdate { update_id: number; message?: TgMessage; callback_query?: CallbackQuery; pre_checkout_query?: { id: string; from: TgUser }; }

type ApiResult<T> = { ok: boolean; result: T; description?: string };

export class Telegram {
  constructor(private readonly token: string) {}
  private async call<T>(method: string, body: Record<string, unknown>): Promise<T> {
    const response = await fetch(`https://api.telegram.org/bot${this.token}/${method}`, {
      method: "POST", headers: { "content-type": "application/json" }, body: JSON.stringify(body),
    });
    const data = await response.json<ApiResult<T>>();
    if (!data.ok) throw new Error(`Telegram ${method}: ${data.description || response.status}`);
    return data.result;
  }
  getMe(): Promise<TgUser> { return this.call("getMe", {}); }
  sendMessage(chat_id: number, text: string, reply_markup?: Keyboard): Promise<TgMessage> {
    return this.call("sendMessage", { chat_id, text, parse_mode: "HTML", reply_markup });
  }
  editText(chat_id: number, message_id: number, text: string, reply_markup?: Keyboard): Promise<unknown> {
    return this.call("editMessageText", { chat_id, message_id, text, parse_mode: "HTML", reply_markup });
  }
  editMarkup(chat_id: number, message_id: number, reply_markup?: Keyboard): Promise<unknown> {
    return this.call("editMessageReplyMarkup", { chat_id, message_id, reply_markup });
  }
  deleteMessage(chat_id: number, message_id: number): Promise<unknown> {
    return this.call("deleteMessage", { chat_id, message_id });
  }
  answerCallbackQuery(callback_query_id: string, value: { text?: string; show_alert?: boolean } = {}): Promise<unknown> {
    return this.call("answerCallbackQuery", { callback_query_id, ...value });
  }
  answerPreCheckoutQuery(id: string, ok = true): Promise<unknown> {
    return this.call("answerPreCheckoutQuery", { pre_checkout_query_id: id, ok });
  }
  sendInvoice(chat_id: number, title: string, description: string, payload: string, amount: number): Promise<unknown> {
    return this.call("sendInvoice", { chat_id, title, description, payload, currency: "XTR", prices: [{ label: title, amount }] });
  }
  async sendPhoto(chatId: number, bytes: Uint8Array, caption: string): Promise<unknown> {
    const form = new FormData();
    form.set("chat_id", String(chatId)); form.set("caption", caption);
    form.set("photo", new Blob([new Uint8Array(bytes).buffer as ArrayBuffer], { type: "image/png" }), "post.png");
    const response = await fetch(`https://api.telegram.org/bot${this.token}/sendPhoto`, { method: "POST", body: form });
    const data = await response.json<ApiResult<unknown>>();
    if (!data.ok) throw new Error(`Telegram sendPhoto: ${data.description || response.status}`);
    return data.result;
  }
}

export const kb = (inline_keyboard: InlineButton[][]): Keyboard => ({ inline_keyboard });


### `src/threads.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/threads.ts
import { launch, type BrowserContext, type Page } from "@cloudflare/playwright";
import { LIMITS, type Env } from "./config";
import { cleanPostText } from "./i18n";

export interface Post { text:string; has_image:boolean; has_video:boolean; image?:Uint8Array }
export interface Comment { author:string; text:string; top?:number }
type Status = "ok"|"user_not_found"|"session_expired"|"no_posts"|"post_not_found"|"all_dead";
type Account = {name:string;cookies:string;hourly_requests:number;hourly_reset:string};
const BASE = (env:Env) => env.BASE_URL || "https://www.threads.com";
const iso=()=>new Date().toISOString();

const COLLECT_POSTS = `() => {
 const posts=[],seen=new Set();
 const containers=document.querySelectorAll('div[data-pressable-container="true"],article,div[role="article"]');
 for(const container of containers){let bestText='',has_image=false,has_video=false;
  for(const img of container.querySelectorAll('img[src*="cdninstagram.com"],img[src*="fbcdn.net"]')){if((img.naturalWidth||img.width||0)>200){has_image=true;break}}
  has_video=container.querySelectorAll('video,div[role="button"] svg[aria-label*="video"],div[role="button"] svg[aria-label="Play"]').length>0;
  for(const el of container.querySelectorAll('span[dir="auto"],div[dir="auto"],span[class*="x1lliihq"]')){const text=(el.innerText||'').trim();if(text.length<20||/^(Follow|Подписаться|Translate|Перевести|See translation|See more|Like|Reply|Repost|Share|Verified|Автор|Ещё|Нравится|Поделиться)/i.test(text)||/^\\d+$/.test(text)||/^\\d{1,2}\\s*[hчдms]$/i.test(text))continue;if(text.length>bestText.length)bestText=text}
  if(bestText.length>25||has_image||has_video){const key=bestText.substring(0,100)+(has_image?'_img':'')+(has_video?'_vid':'');if(!seen.has(key)){seen.add(key);posts.push({text:bestText,has_image,has_video})}}
 } return posts;
}`;

function cookies(raw:string): any[] {
 return (JSON.parse(raw) as Record<string,unknown>[]).map(c=>{let sameSite=String(c.sameSite||"Lax");if(["unspecified","null",""] .includes(sameSite))sameSite="Lax";else if(["no_restriction","none"].includes(sameSite.toLowerCase()))sameSite="None";else sameSite=sameSite[0].toUpperCase()+sameSite.slice(1).toLowerCase();const out:any={name:c.name,value:c.value,domain:c.domain,path:c.path||"/",httpOnly:Boolean(c.httpOnly),secure:c.secure!==false,sameSite};const exp=c.expirationDate??c.expires;if(typeof exp==="number"&&exp>0)out.expires=exp;return out});
}

async function collectPosts(page:Page,target=20):Promise<Post[]>{const all:Post[]=[],seen=new Set<string>();let stall=0;for(let i=0;i<35;i++){const current=await page.evaluate(COLLECT_POSTS) as Post[];let added=0;for(const post of current){const value={...post,text:cleanPostText(post.text)};if(value.text.length<15&&!value.has_image&&!value.has_video)continue;const key=value.text.slice(0,120)+(value.has_image?'_img':'')+(value.has_video?'_vid':'');if(!seen.has(key)){seen.add(key);all.push(value);added++}}if(all.length>=target)break;stall=added?0:stall+1;if(stall>=6)break;await page.evaluate(()=>window.scrollBy(0,1100));await new Promise(r=>setTimeout(r,2300))}return all.slice(0,target)}

async function checkProfile(page:Page,env:Env,username:string):Promise<Status|null>{await page.goto(`${BASE(env)}/@${username}`,{waitUntil:"domcontentloaded",timeout:30_000});await new Promise(r=>setTimeout(r,4000));const body=await page.locator("body").innerText().catch(()=>"");if(["Page not found","Страница не найдена","isn't available","недоступна"].some(x=>body.includes(x)))return "user_not_found";if(page.url().includes("login"))return "session_expired";await page.waitForSelector("span[dir='auto'],div[dir='auto']",{timeout:15_000}).catch(()=>{});return null}

async function chooseAccount(env:Env,tried:string[]):Promise<Account|null>{const cutoff=new Date(Date.now()-3600_000).toISOString();await env.DB.prepare("UPDATE threads_accounts SET hourly_requests=0,hourly_reset=? WHERE hourly_reset<?").bind(iso(),cutoff).run();const marks=tried.map(()=>'?').join(',');const q=`SELECT name,cookies,hourly_requests,hourly_reset FROM threads_accounts WHERE enabled=1 AND is_alive=1 AND hourly_requests<?${tried.length?` AND name NOT IN (${marks})`:''} ORDER BY hourly_requests ASC,RANDOM() LIMIT 1`;return await env.DB.prepare(q).bind(LIMITS.accountHourly,...tried).first<Account>();}
async function mark(env:Env,name:string,ok:boolean,error="",posts=0,cookieData?:string){if(ok)await env.DB.prepare("UPDATE threads_accounts SET is_alive=1,last_error=NULL,requests_count=requests_count+1,posts_sent=posts_sent+?,hourly_requests=hourly_requests+1,last_used=?,updated_at=?,cookies=COALESCE(?,cookies) WHERE name=?").bind(posts,iso(),iso(),cookieData||null,name).run();else await env.DB.prepare("UPDATE threads_accounts SET is_alive=0,last_error=?,errors_count=errors_count+1,updated_at=? WHERE name=?").bind(error.slice(0,500),iso(),name).run()}

async function contextFor(env:Env,account:Account):Promise<{browser:any;context:BrowserContext;page:Page}>{const browser=await launch(env.BROWSER);const context=await browser.newContext({userAgent:"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",viewport:{width:680,height:900}});await context.addCookies(cookies(account.cookies));const page=await context.newPage();return{browser,context,page}}

async function capturePosts(page:Page, posts:Post[]):Promise<Post[]>{await page.evaluate(()=>{window.scrollTo(0,0);document.querySelectorAll('div[role="dialog"]').forEach(e=>e.remove());document.querySelectorAll('nav,header').forEach((e:any)=>e.style.display='none')});const result:Post[]=[];for(const post of posts){try{const handle=await page.evaluateHandle((search:string)=>{let nodes=Array.from(document.querySelectorAll('article,div[role="article"]'));if(!nodes.length)nodes=Array.from(document.querySelectorAll('div[data-pressable-container="true"]'));return nodes.find(node=>Array.from(node.querySelectorAll('span[dir="auto"],div[dir="auto"]')).some((b:any)=>(b.innerText||'').trim().startsWith(search)))||null},post.text.slice(0,50));const element=handle.asElement();if(element){await element.scrollIntoViewIfNeeded();const shot=await element.screenshot({type:"png"});result.push({...post,image:new Uint8Array(shot)})}await handle.dispose()}catch{continue}}return result}

export async function fetchPosts(env:Env,username:string,mode:"text"|"img",amount=20):Promise<{data:Post[]|null;status:Status;account?:string}>{const tried:string[]=[];while(true){const acc=await chooseAccount(env,tried);if(!acc)return{data:null,status:"all_dead"};tried.push(acc.name);let opened:any;try{opened=await contextFor(env,acc);const invalid=await checkProfile(opened.page,env,username);if(invalid){if(invalid==="session_expired"){await mark(env,acc.name,false,"Session expired");continue}return{data:null,status:invalid,account:acc.name}}let data=await collectPosts(opened.page,amount);if(!data.length)return{data:null,status:"no_posts",account:acc.name};if(mode==="img")data=await capturePosts(opened.page,data);const updated=JSON.stringify(await opened.context.cookies());await mark(env,acc.name,true,"",data.length,updated);return{data,status:"ok",account:acc.name}}catch(error){await mark(env,acc.name,false,error instanceof Error?error.message:String(error));continue}finally{await opened?.browser.close().catch(()=>{})}}}

async function collectComments(page:Page,target=20):Promise<Comment[]>{const result:Comment[]=[],seen=new Set<string>();let stall=0;await new Promise(r=>setTimeout(r,3000));for(let attempt=0;attempt<20;attempt++){const current=await page.evaluate(() => {const out:any[]=[];for(const container of document.querySelectorAll('div[data-pressable-container="true"]')){const top=(container as HTMLElement).getBoundingClientRect().top+window.scrollY;const link=container.querySelector('a[href^="/@"][role="link"]');const match=(link?.getAttribute('href')||'').match(/\/@([A-Za-z0-9._]+)/);const author=match?'@'+match[1].toLowerCase():'—';let text='';for(const sp of container.querySelectorAll('span[dir="auto"]')){const t=((sp as HTMLElement).innerText||'').trim();if(!t||t.length<3||t.toLowerCase()===author.replace('@','')||/^(Follow|Подписаться|Translate|Перевести|Reply|Ответ|Repost|Share|Send|Like|More|Verified|See translation|Автор|Author|Ещё|Нравится|Поделиться)$/i.test(t)||/^\d+$/.test(t)||/^\d+\s*[hHчмсmsdд]$/.test(t))continue;if(t.length>text.length)text=t}if(text&&/[A-Za-zА-Яа-яÀ-ÿ\u0400-\u04FF\u4e00-\u9fff\u3040-\u30ff]/.test(text))out.push({author,text,top})}out.sort((a,b)=>a.top-b.top);return out.slice(1)}) as Comment[];let added=0;for(const c of current){c.text=cleanPostText(c.text);const key=(c.author+'|'+c.text.slice(0,120)).toLowerCase();if(!seen.has(key)){seen.add(key);result.push(c);added++}}result.sort((a,b)=>(a.top||0)-(b.top||0));if(result.length>=target)break;stall=added?0:stall+1;if(stall>=3)break;await page.evaluate(()=>window.scrollBy(0,1200));await new Promise(r=>setTimeout(r,2500))}return result.slice(0,target)}

export async function fetchComments(env:Env,username:string,index:number,amount=20):Promise<{data:Comment[]|null;status:Status;account?:string}>{const tried:string[]=[];while(true){const acc=await chooseAccount(env,tried);if(!acc)return{data:null,status:"all_dead"};tried.push(acc.name);let opened:any;try{opened=await contextFor(env,acc);const invalid=await checkProfile(opened.page,env,username);if(invalid){if(invalid==="session_expired"){await mark(env,acc.name,false,"Session expired");continue}return{data:null,status:invalid,account:acc.name}}const posts=await collectPosts(opened.page,index+3);if(index>=posts.length)return{data:null,status:"post_not_found",account:acc.name};const search=posts[index].text.slice(0,50);const href=await opened.page.evaluate((s:string)=>{let nodes=Array.from(document.querySelectorAll('article,div[role="article"],div[data-pressable-container="true"]'));for(const post of nodes){if(Array.from(post.querySelectorAll('span[dir="auto"],div[dir="auto"]')).some((b:any)=>(b.innerText||'').trim().startsWith(s))){const a=post.querySelector('a[href*="/post/"]');if(a)return a.getAttribute('href')}}return null},search);if(!href)return{data:null,status:"post_not_found",account:acc.name};await opened.page.goto(href.startsWith('/')?BASE(env)+href:href,{waitUntil:"domcontentloaded",timeout:30_000});await new Promise(r=>setTimeout(r,4000));await opened.page.evaluate(()=>window.scrollBy(0,800));const data=await collectComments(opened.page,amount);const updated=JSON.stringify(await opened.context.cookies());await mark(env,acc.name,true,"",data.length,updated);return{data,status:"ok",account:acc.name}}catch(error){await mark(env,acc.name,false,error instanceof Error?error.message:String(error));continue}finally{await opened?.browser.close().catch(()=>{})}}}

/** Re-opens dead sessions, mirroring the Python bot's reload/periodic health check. */
export async function reviveAccounts(env:Env):Promise<{name:string;ok:boolean}[]>{const rows=(await env.DB.prepare("SELECT name,cookies,hourly_requests,hourly_reset FROM threads_accounts WHERE enabled=1 AND is_alive=0 ORDER BY name").all<Account>()).results,out:{name:string;ok:boolean}[]=[];for(const account of rows){let opened:any;try{opened=await contextFor(env,account);await opened.page.goto(`${BASE(env)}/`,{waitUntil:"domcontentloaded",timeout:30_000});await new Promise(r=>setTimeout(r,3000));const ok=!opened.page.url().includes("login");if(ok){const updated=JSON.stringify(await opened.context.cookies());await env.DB.prepare("UPDATE threads_accounts SET is_alive=1,last_error=NULL,cookies=?,updated_at=? WHERE name=?").bind(updated,iso(),account.name).run()}else await mark(env,account.name,false,"Session expired");out.push({name:account.name,ok})}catch(error){await mark(env,account.name,false,error instanceof Error?error.message:String(error));out.push({name:account.name,ok:false})}finally{await opened?.browser.close().catch(()=>{})}}return out}


### `src/translations.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/translations.ts
// Generated from the original Python bot; keep user-facing copy unchanged.
export const TRANSLATIONS = {
  "ru": {
    "welcome": "👋 <b>Добро пожаловать в Threads Reader Bot!</b>\n\nЯ помогаю читать посты и комментарии из Threads.\n\n📌 <b>Как пользоваться:</b>\n1. Напиши username\n2. Выбери формат — текст или скриншоты\n3. Ответь на пост, чтобы загрузить комментарии\n\nПример: <code>zuck</code>\n\n{status}\n\nИспользуй /help, чтобы посмотреть доступные команды.",
    "help": "ℹ️ <b>Как пользоваться:</b>\n\n1. Напиши username\n2. Выбери формат\n3. Листай кнопкой «Ещё»\n4. Ответь на пост — комментарии\n\n/subscribe — подписка\n/status — статус\n/support — поддержка",
    "select_language": "🌍 Выберите язык:",
    "language_set": "✅ Язык установлен: Русский",
    "subscription_active": "✅ Подписка активна ({days} дн.)",
    "free_limit": "🆓 Бесплатно: {daily} на сегодня, {monthly} в этом месяце",
    "no_subscription": "❌ Нет активной подписки",
    "daily_limit_reached": "⏳ Дневной лимит ({limit}/{limit}) исчерпан.\nНовые попытки появятся завтра!",
    "monthly_limit_reached": "🛑 Месячный лимит бесплатных запросов исчерпан.",
    "posts_found": "📱 <b>@{username}</b> ({count} постов)",
    "invalid_username": "❌ Это не username.",
    "rate_limit": "⏳ {reason}",
    "no_accounts": "❌ Нет рабочих аккаунтов",
    "admins_notified": "Админы оповещены.",
    "all_dead": "❌ Все аккаунты недоступны.",
    "user_not_found": "❌ @{username} не найден.",
    "no_posts": "😔 Нет постов.",
    "no_more": "📭 Больше нет.",
    "photo": "фото",
    "video": "видео",
    "subscribe_btn": "💳 Оформить подписку",
    "subscribe_unlocks": "🔓 Подписка даёт полный безлимит:",
    "text_btn": "📝 Текст",
    "screens_btn": "🖼 Скрины",
    "format": "формат",
    "more": "Ещё",
    "of": "из",
    "all": "Все",
    "more_available": "➡️ Есть ещё!",
    "reply_for_comments": "💬 Ответь на пост — комментарии",
    "send_other": "Нужен другой username? Пиши👇",
    "paid": "Оплачено",
    "until": "До",
    "not_found": "Не найдена",
    "pay": "Оплатить",
    "i_paid": "Я оплатил",
    "after_payment": "После оплаты нажми",
    "subscription": "Подписка",
    "days": "дней",
    "unlimited": "Безлимит",
    "left_today": "Осталось сегодня",
    "renew": "Продлить",
    "subscribe": "Оформить",
    "active": "Активна",
    "no_sub_short": "Нет подписки",
    "support_title": "🆘 Поддержка",
    "ask_question": "Задай вопрос",
    "suggest_idea": "Предложи идею",
    "choose": "Выбери",
    "question": "Вопрос",
    "suggestion": "Предложение",
    "my_tickets": "Мои обращения",
    "payment_method": "💳 Способ оплаты:"
  },
  "en": {
    "welcome": "👋 <b>Welcome to Threads Reader Bot!</b>\n\nI help you read posts and comments from Threads.\n\n📌 <b>How to use:</b>\n1. Send username\n2. Choose format — text or screenshots\n3. Reply to a post to load comments\n\nExample: <code>zuck</code>\n\n{status}\n\nUse /help to see available commands.",
    "help": "ℹ️ <b>How to use:</b>\n\n1. Send username\n2. Choose format\n3. Use «More» button\n4. Reply to a post for comments\n\n/subscribe — subscription\n/status — status\n/support — support",
    "select_language": "🌍 Select language:",
    "language_set": "✅ Language set: English",
    "subscription_active": "✅ Subscription active ({days} days)",
    "free_limit": "🆓 Free: {daily} today, {monthly} this month",
    "no_subscription": "❌ No active subscription",
    "daily_limit_reached": "⏳ Daily limit ({limit}/{limit}) reached.\nNew attempts tomorrow!",
    "monthly_limit_reached": "🛑 Monthly free limit reached.",
    "posts_found": "📱 <b>@{username}</b> ({count} posts)",
    "invalid_username": "❌ Invalid username.",
    "rate_limit": "⏳ {reason}",
    "no_accounts": "❌ No working accounts",
    "admins_notified": "Admins notified.",
    "all_dead": "❌ All accounts unavailable.",
    "user_not_found": "❌ @{username} not found.",
    "no_posts": "😔 No posts.",
    "no_more": "📭 No more.",
    "photo": "photo",
    "video": "video",
    "subscribe_btn": "💳 Subscribe",
    "subscribe_unlocks": "🔓 Subscription unlocks unlimited access:",
    "text_btn": "📝 Text",
    "screens_btn": "🖼 Screens",
    "format": "format",
    "more": "More",
    "of": "of",
    "all": "All",
    "more_available": "➡️ More available!",
    "reply_for_comments": "💬 Reply to post for comments",
    "send_other": "Need another username? Type it👇",
    "paid": "Paid",
    "until": "Until",
    "not_found": "Not found",
    "pay": "Pay",
    "i_paid": "I paid",
    "after_payment": "After payment click",
    "subscription": "Subscription",
    "days": "days",
    "unlimited": "Unlimited",
    "left_today": "Left today",
    "renew": "Renew",
    "subscribe": "Subscribe",
    "active": "Active",
    "no_sub_short": "No subscription",
    "support_title": "🆘 Support",
    "ask_question": "Ask a question",
    "suggest_idea": "Suggest an idea",
    "choose": "Select",
    "question": "Question",
    "suggestion": "Suggestion",
    "my_tickets": "My tickets",
    "payment_method": "💳 Payment method:"
  },
  "de": {
    "welcome": "👋 <b>Willkommen beim Threads Reader Bot!</b>\n\nIch helfe dir, Beiträge und Kommentare aus Threads zu lesen.\n\n📌 <b>Wie benutzen:</b>\n1. Sende einen Benutzernamen\n2. Wähle Format — Text oder Screenshots\n3. Antworte auf einen Beitrag, um Kommentare zu laden\n\nBeispiel: <code>zuck</code>\n\n{status}\n\nVerwende /help, um verfügbare Befehle zu sehen.",
    "help": "ℹ️ <b>Wie benutzen:</b>\n\n1. Sende einen Benutzernamen\n2. Wähle Format\n3. Verwende «Mehr»\n4. Antworte auf einen Beitrag — Kommentare\n\n/subscribe — Abo\n/status — Status\n/support — Support",
    "select_language": "🌍 Sprache wählen:",
    "language_set": "✅ Sprache: Deutsch",
    "subscription_active": "✅ Abo aktiv ({days} Tage)",
    "free_limit": "🆓 Kostenlos: {daily} heute, {monthly} diesen Monat",
    "no_subscription": "❌ Kein aktives Abo",
    "daily_limit_reached": "⏳ Tageslimit ({limit}/{limit}) erreicht.\nMorgen neue Versuche!",
    "monthly_limit_reached": "🛑 Monatliches Gratislimit erreicht.",
    "posts_found": "📱 <b>@{username}</b> ({count} Beiträge)",
    "invalid_username": "❌ Ungültiger Benutzername.",
    "rate_limit": "⏳ {reason}",
    "no_accounts": "❌ Keine funktionierenden Konten",
    "admins_notified": "Admins benachrichtigt.",
    "all_dead": "❌ Alle Konten nicht verfügbar.",
    "user_not_found": "❌ @{username} nicht gefunden.",
    "no_posts": "😔 Keine Beiträge.",
    "no_more": "📭 Keine weiteren.",
    "photo": "Foto",
    "video": "Video",
    "subscribe_btn": "💳 Abonnieren",
    "subscribe_unlocks": "🔓 Abo gibt unbegrenzten Zugang:",
    "text_btn": "📝 Text",
    "screens_btn": "🖼 Screenshots",
    "format": "Format",
    "more": "Mehr",
    "of": "von",
    "all": "Alle",
    "more_available": "➡️ Mehr verfügbar!",
    "reply_for_comments": "💬 Antworte auf den Beitrag für Kommentare",
    "send_other": "Anderen Benutzernamen? Schreib👇",
    "paid": "Bezahlt",
    "until": "Bis",
    "not_found": "Nicht gefunden",
    "pay": "Bezahlen",
    "i_paid": "Ich habe bezahlt",
    "after_payment": "Nach Zahlung klicken",
    "subscription": "Abo",
    "days": "Tage",
    "unlimited": "Unbegrenzt",
    "left_today": "Heute übrig",
    "renew": "Verlängern",
    "subscribe": "Abonnieren",
    "active": "Aktiv",
    "no_sub_short": "Kein Abo",
    "support_title": "🆘 Support",
    "ask_question": "Stelle eine Frage",
    "suggest_idea": "Schlage eine Idee vor",
    "choose": "Wähle",
    "question": "Frage",
    "suggestion": "Vorschlag",
    "my_tickets": "Meine Tickets",
    "payment_method": "💳 Zahlungsmethode:"
  },
  "es": {
    "welcome": "👋 <b>¡Bienvenido a Threads Reader Bot!</b>\n\nTe ayudo a leer publicaciones y comentarios de Threads.\n\n📌 <b>Cómo usar:</b>\n1. Envía un username\n2. Elige formato — texto o capturas\n3. Responde a un post para cargar comentarios\n\nEjemplo: <code>zuck</code>\n\n{status}\n\nUsa /help para ver comandos disponibles.",
    "help": "ℹ️ <b>Cómo usar:</b>\n\n1. Envía un username\n2. Elige formato\n3. Usa el botón «Más»\n4. Responde a un post — comentarios\n\n/subscribe — suscripción\n/status — estado\n/support — soporte",
    "select_language": "🌍 Selecciona idioma:",
    "language_set": "✅ Idioma: Español",
    "subscription_active": "✅ Suscripción activa ({days} días)",
    "free_limit": "🆓 Gratis: {daily} hoy, {monthly} este mes",
    "no_subscription": "❌ Sin suscripción activa",
    "daily_limit_reached": "⏳ Límite diario ({limit}/{limit}) alcanzado.\n¡Nuevos intentos mañana!",
    "monthly_limit_reached": "🛑 Límite gratuito mensual alcanzado.",
    "posts_found": "📱 <b>@{username}</b> ({count} posts)",
    "invalid_username": "❌ Username inválido.",
    "rate_limit": "⏳ {reason}",
    "no_accounts": "❌ Sin cuentas funcionales",
    "admins_notified": "Admins notificados.",
    "all_dead": "❌ Todas las cuentas no disponibles.",
    "user_not_found": "❌ @{username} no encontrado.",
    "no_posts": "😔 Sin publicaciones.",
    "no_more": "📭 No hay más.",
    "photo": "foto",
    "video": "vídeo",
    "subscribe_btn": "💳 Suscribirse",
    "subscribe_unlocks": "🔓 La suscripción da acceso ilimitado:",
    "text_btn": "📝 Texto",
    "screens_btn": "🖼 Capturas",
    "format": "formato",
    "more": "Más",
    "of": "de",
    "all": "Todo",
    "more_available": "➡️ ¡Hay más!",
    "reply_for_comments": "💬 Responde al post para comentarios",
    "send_other": "¿Otro username? Escríbelo👇",
    "paid": "Pagado",
    "until": "Hasta",
    "not_found": "No encontrado",
    "pay": "Pagar",
    "i_paid": "He pagado",
    "after_payment": "Después del pago haz clic",
    "subscription": "Suscripción",
    "days": "días",
    "unlimited": "Ilimitado",
    "left_today": "Restante hoy",
    "renew": "Renovar",
    "subscribe": "Suscribirse",
    "active": "Activa",
    "no_sub_short": "Sin suscripción",
    "support_title": "🆘 Soporte",
    "ask_question": "Haz una pregunta",
    "suggest_idea": "Sugiere una idea",
    "choose": "Elige",
    "question": "Pregunta",
    "suggestion": "Sugerencia",
    "my_tickets": "Mis tickets",
    "payment_method": "💳 Método de pago:"
  },
  "pt": {
    "welcome": "👋 <b>Bem-vindo ao Threads Reader Bot!</b>\n\nAjudo você a ler posts e comentários do Threads.\n\n📌 <b>Como usar:</b>\n1. Envie um username\n2. Escolha formato — texto ou screenshots\n3. Responda a um post para carregar comentários\n\nExemplo: <code>zuck</code>\n\n{status}\n\nUse /help para ver os comandos disponíveis.",
    "help": "ℹ️ <b>Como usar:</b>\n\n1. Envie um username\n2. Escolha formato\n3. Use o botão «Mais»\n4. Responda a um post — comentários\n\n/subscribe — assinatura\n/status — status\n/support — suporte",
    "select_language": "🌍 Selecione o idioma:",
    "language_set": "✅ Idioma: Português",
    "subscription_active": "✅ Assinatura ativa ({days} dias)",
    "free_limit": "🆓 Grátis: {daily} hoje, {monthly} este mês",
    "no_subscription": "❌ Sem assinatura ativa",
    "daily_limit_reached": "⏳ Limite diário ({limit}/{limit}) atingido.\nNovas tentativas amanhã!",
    "monthly_limit_reached": "🛑 Limite mensal gratuito atingido.",
    "posts_found": "📱 <b>@{username}</b> ({count} posts)",
    "invalid_username": "❌ Username inválido.",
    "rate_limit": "⏳ {reason}",
    "no_accounts": "❌ Sem contas funcionais",
    "admins_notified": "Admins notificados.",
    "all_dead": "❌ Todas as contas indisponíveis.",
    "user_not_found": "❌ @{username} não encontrado.",
    "no_posts": "😔 Sem posts.",
    "no_more": "📭 Não há mais.",
    "photo": "foto",
    "video": "vídeo",
    "subscribe_btn": "💳 Assinar",
    "subscribe_unlocks": "🔓 A assinatura dá acesso ilimitado:",
    "text_btn": "📝 Texto",
    "screens_btn": "🖼 Screenshots",
    "format": "formato",
    "more": "Mais",
    "of": "de",
    "all": "Tudo",
    "more_available": "➡️ Há mais!",
    "reply_for_comments": "💬 Responda ao post para comentários",
    "send_other": "Outro username? Escreva👇",
    "paid": "Pago",
    "until": "Até",
    "not_found": "Não encontrado",
    "pay": "Pagar",
    "i_paid": "Eu paguei",
    "after_payment": "Após o pagamento, clique",
    "subscription": "Assinatura",
    "days": "dias",
    "unlimited": "Ilimitado",
    "left_today": "Restante hoje",
    "renew": "Renovar",
    "subscribe": "Assinar",
    "active": "Ativa",
    "no_sub_short": "Sem assinatura",
    "support_title": "🆘 Suporte",
    "ask_question": "Faça uma pergunta",
    "suggest_idea": "Sugira uma ideia",
    "choose": "Selecione",
    "question": "Pergunta",
    "suggestion": "Sugestão",
    "my_tickets": "Meus tickets",
    "payment_method": "💳 Método de pagamento:"
  }
} as const;

export const LANGUAGE_NAMES = {
  "ru": "🇷🇺 Русский",
  "en": "🇬🇧 English",
  "de": "🇩🇪 Deutsch",
  "es": "🇪🇸 Español",
  "pt": "🇵🇹 Português"
} as const;


### `src/bot.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/src/bot.ts
import { adminIds, isAdmin, LIMITS, type Env } from "./config";
import { Database } from "./db";
import { LANGUAGE_NAMES, languages, text } from "./i18n";
import { fetchComments, fetchPosts, reviveAccounts, type Comment, type Post } from "./threads";
import { kb, Telegram, type CallbackQuery, type TelegramUpdate, type TgMessage } from "./telegram";

const esc=(v:unknown)=>String(v??"").replaceAll("&","&amp;").replaceAll("<","&lt;").replaceAll(">","&gt;");
const fmtDate=(d:Date)=>`${String(d.getUTCDate()).padStart(2,'0')}.${String(d.getUTCMonth()+1).padStart(2,'0')}.${d.getUTCFullYear()}`;

export class Bot {
 private db:Database; private tg:Telegram;
 constructor(private env:Env){this.db=new Database(env);this.tg=new Telegram(env.TELEGRAM_TOKEN)}
 private async lang(uid:number){return this.db.getLang(uid)}
 private async removeButtons(cid:number){const id=Number(await this.db.state(cid,"last_button"));if(id){await this.tg.editMarkup(cid,id).catch(()=>{});await this.db.clearState(cid,"last_button")}}
 private async buttons(cid:number,value:string,keyboard:ReturnType<typeof kb>){await this.removeButtons(cid);const msg=await this.tg.sendMessage(cid,value,keyboard);await this.db.setState(cid,"last_button",String(msg.message_id))}
 private async access(uid:number){const lang=await this.lang(uid);if(isAdmin(this.env,uid)||await this.db.hasSubscription(uid))return{ok:true,note:"",kind:"ok"};const u=await this.db.usage(uid);if(u.monthly>=LIMITS.freeMonthly)return{ok:false,note:text("monthly_limit_reached",lang),kind:"month_limit"};if(u.daily>=LIMITS.freeDaily)return{ok:false,note:text("daily_limit_reached",lang,{limit:LIMITS.freeDaily}),kind:"day_limit"};return{ok:true,note:text("free_limit",lang,{daily:LIMITS.freeDaily-u.daily,monthly:LIMITS.freeMonthly-u.monthly}),kind:"ok"}}
 private async alert(value:string){for(const id of adminIds(this.env))await this.tg.sendMessage(id,`🚨 <b>ALERT</b>\n\n${esc(value)}`).catch(()=>{})}

 async update(update:TelegramUpdate){if(update.pre_checkout_query){await this.tg.answerPreCheckoutQuery(update.pre_checkout_query.id);return}if(update.callback_query){await this.callback(update.callback_query);return}if(update.message)await this.message(update.message)}
 private async message(m:TgMessage){if(!m.from)return;const uid=m.from.id,cid=m.chat.id,raw=(m.text||m.caption||"").trim();if(await this.db.isBanned(uid))return;
  if(m.successful_payment){const exp=await this.db.activate(uid,"stars",m.successful_payment.total_amount);await this.db.logEvent(uid,"subscribe","stars");await this.tg.sendMessage(cid,`🎉 <b>${text("paid",await this.lang(uid))}!</b> ${text("until",await this.lang(uid))} ${fmtDate(exp)}`);return}
  if(m.reply_to_message?.from?.is_bot){await this.replyComments(m);return}
  const waiting=await this.db.state(uid,"waiting_support");if(waiting){await this.supportInput(m,waiting);return}
  const adminReply=await this.db.state(uid,"admin_reply");if(adminReply&&isAdmin(this.env,uid)){await this.adminReply(m,Number(adminReply));return}
  if(raw.startsWith("/")){const [command]=raw.split(/\s+/);switch(command.split("@")[0]){case"/start":return this.start(m);case"/language":return this.language(m);case"/help":return this.help(m);case"/subscribe":return this.subscribe(m);case"/status":return this.status(m);case"/support":return this.support(m);case"/admin":return this.admin(m);case"/ban":return this.ban(m,true);case"/unban":return this.ban(m,false);default:return}}
  if(isAdmin(this.env,uid)&&/^\d+\s+.+/s.test(raw)){const match=raw.match(/^(\d+)\s+(.+)$/s)!;await this.answerTicket(cid,Number(match[1]),match[2]);return}
  await this.username(m,raw);
 }
 private async start(m:TgMessage){const uid=m.from!.id,cid=m.chat.id;if(!await this.db.hasLang(uid)){let lang=(m.from!.language_code||"ru").slice(0,2);if(!languages.includes(lang as any))lang="en";await this.db.setLang(uid,lang);await this.tg.sendMessage(cid,"🌍 Выберите язык / Select language / Sprache wählen:",this.languageKb());return}await this.removeButtons(cid);await this.db.logEvent(uid,"start");const lang=await this.lang(uid),sub=await this.db.subscription(uid),u=await this.db.usage(uid);const status=sub?.active?text("subscription_active",lang,{days:Number(sub.days_left)}):text("free_limit",lang,{daily:Math.max(0,LIMITS.freeDaily-u.daily),monthly:Math.max(0,LIMITS.freeMonthly-u.monthly)});await this.tg.sendMessage(cid,text("welcome",lang,{status}))}
 private languageKb(){return kb([[{text:LANGUAGE_NAMES.ru,callback_data:"set_lang:ru"},{text:LANGUAGE_NAMES.en,callback_data:"set_lang:en"}],[{text:LANGUAGE_NAMES.de,callback_data:"set_lang:de"},{text:LANGUAGE_NAMES.es,callback_data:"set_lang:es"}],[{text:LANGUAGE_NAMES.pt,callback_data:"set_lang:pt"}]])}
 private async language(m:TgMessage){await this.removeButtons(m.chat.id);await this.tg.sendMessage(m.chat.id,text("select_language",await this.lang(m.from!.id)),this.languageKb())}
 private async help(m:TgMessage){await this.removeButtons(m.chat.id);let value=text("help",await this.lang(m.from!.id));if(isAdmin(this.env,m.from!.id))value+="\n\n🔐 /admin";await this.tg.sendMessage(m.chat.id,value)}
 private async subscribe(m:TgMessage){const uid=m.from!.id,cid=m.chat.id,lang=await this.lang(uid),sub=await this.db.subscription(uid);if(sub?.active){await this.buttons(cid,`✅ ${text("until",lang)} ${String(sub.expires_at).slice(0,10)} (${sub.days_left} ${text("days",lang)})`,kb([[{text:`🔄 ${text("renew",lang)}`,callback_data:"sub:choose"}]]));return}const u=await this.db.usage(uid);await this.buttons(cid,`📱 <b>${text("subscription",lang)} — 30 ${text("days",lang)}</b>\n✅ ${text("unlimited",lang)}\n🆓 ${text("left_today",lang)}: ${Math.max(0,LIMITS.freeDaily-u.daily)}`,kb([[{text:`💳 ${text("subscribe",lang)}`,callback_data:"sub:choose"}]]))}
 private async status(m:TgMessage){const uid=m.from!.id,lang=await this.lang(uid),sub=await this.db.subscription(uid);if(sub?.active)await this.tg.sendMessage(m.chat.id,`✅ <b>${text("active",lang)}</b> ${text("until",lang)} ${String(sub.expires_at).slice(0,10)} (${sub.days_left} ${text("days",lang)})`);else{const u=await this.db.usage(uid);await this.tg.sendMessage(m.chat.id,`❌ <b>${text("no_sub_short",lang)}</b>\n🆓 ${text("left_today",lang)}: ${Math.max(0,LIMITS.freeDaily-u.daily)}\n/subscribe`)}}
 private async support(m:TgMessage){const lang=await this.lang(m.from!.id);await this.buttons(m.chat.id,`<b>${text("support_title",lang)}</b>\n\n• ${text("ask_question",lang)}\n• ${text("suggest_idea",lang)}\n\n${text("choose",lang)} 👇`,kb([[{text:`❓ ${text("question",lang)}`,callback_data:"sup:write:question"}],[{text:`💡 ${text("suggestion",lang)}`,callback_data:"sup:write:suggestion"}],[{text:`📋 ${text("my_tickets",lang)}`,callback_data:"sup:my"}]]))}
 private async supportInput(m:TgMessage,type:string){const value=(m.text||m.caption||"").trim();if(!value){await this.tg.sendMessage(m.chat.id,"❌ Сообщение не может быть пустым.");return}await this.db.clearState(m.from!.id,"waiting_support");const id=await this.db.createTicket(m.from!.id,m.from!.username||String(m.from!.id),value,type);await this.db.logEvent(m.from!.id,"ticket",type);const lang=await this.lang(m.from!.id),label=text(type==="suggestion"?"suggestion":"question",lang);await this.tg.sendMessage(m.chat.id,`✅ <b>${label} #${id}</b>\n\nВаше обращение принято. Спасибо!`);for(const aid of adminIds(this.env))await this.tg.sendMessage(aid,`🆘 ${type==="suggestion"?"💡":"❓"} <b>#${id}</b>\n👤 @${esc(m.from!.username||m.from!.id)} (<code>${m.from!.id}</code>)\n📝 ${esc(value.slice(0,500))}`,kb([[{text:"💬 Reply",callback_data:`ticket:reply:${id}`}]] )).catch(()=>{})}
 private async adminReply(m:TgMessage,id:number){await this.db.clearState(m.from!.id,"admin_reply");await this.answerTicket(m.chat.id,id,(m.text||m.caption||"").trim())}
 private async answerTicket(cid:number,id:number,answer:string){const ticket=await this.db.ticket(id);if(!ticket){await this.tg.sendMessage(cid,"❌ Ticket not found.");return}if(!answer){await this.tg.sendMessage(cid,"❌ Empty answer.");return}await this.db.answerTicket(id,answer);await this.tg.sendMessage(Number(ticket.user_id),`💬 <b>Reply to #${id}</b>\n\n${esc(answer)}`).catch(()=>{});await this.tg.sendMessage(cid,`✅ Reply #${id} sent.`)}
 private async ban(m:TgMessage,enabled:boolean){if(!isAdmin(this.env,m.from!.id))return;const p=(m.text||"").split(/\s+/,3),uid=Number(p[1]);if(!Number.isFinite(uid)){await this.tg.sendMessage(m.chat.id,`<code>/${enabled?'ban':'unban'} ID</code>`);return}if(enabled)await this.db.ban(uid,p[2]||"ban");else await this.db.unban(uid);await this.tg.sendMessage(m.chat.id,`${enabled?'🚫':'✅'} ${uid} ${enabled?'banned':'unbanned'}`)}
 private adminKb(open:number){return kb([[{text:"📊 Stats",callback_data:"adm:stats"},{text:"🏥 Health",callback_data:"adm:health"}],[{text:"🔄 Reload",callback_data:"adm:reload"},{text:"📋 Detailed",callback_data:"adm:detailed"}],[{text:"🚫 Bans",callback_data:"adm:banlist"},{text:"👥 Subs",callback_data:"adm:subs"}],[{text:`🆘 Tickets (${open})`,callback_data:"adm:tickets"}],[{text:"📈 Analytics",callback_data:"adm:analytics"}]])}
 private async admin(m:TgMessage){if(!isAdmin(this.env,m.from!.id))return;const counts=await this.db.accountCounts(),subs=await this.db.subscribers(),tickets=await this.db.tickets();await this.buttons(m.chat.id,`🔐 <b>Admin</b>\n🟢${counts.alive||0}/${counts.total} | 👥${subs.length} | 🆘${tickets.length}`,this.adminKb(tickets.length))}
 private async username(m:TgMessage,raw:string){const username=raw.replace(/^@/,""),uid=m.from!.id,lang=await this.lang(uid);if(!/^[\p{L}\p{N}._]{2,}$/u.test(username)){await this.tg.sendMessage(m.chat.id,text("invalid_username",lang));return}const access=await this.access(uid);if(!access.ok){await this.db.logEvent(uid,"free_exhausted",access.kind);await this.buttons(m.chat.id,`${access.note}\n\n<b>${text("subscribe_unlocks",lang)}</b>`,kb([[{text:text("subscribe_btn",lang),callback_data:"sub:choose"}]]));return}const limited=await this.db.rateLimit(uid);if(limited){await this.tg.sendMessage(m.chat.id,`⏳ ${limited}`);return}const counts=await this.db.accountCounts();if(!counts.alive){await this.alert("Все аккаунты мертвы!");await this.tg.sendMessage(m.chat.id,text("no_accounts",lang));return}await this.buttons(m.chat.id,`🔍 <b>@${esc(username)}</b> — ${text("format",lang)}:${access.note?`\n${access.note}`:""}`,kb([[{text:text("text_btn",lang),callback_data:`text:${username}:0`},{text:text("screens_btn",lang),callback_data:`img:${username}:0`}]]))}

 private async callback(cb:CallbackQuery){if(!cb.data||!cb.message)return;const d=cb.data,uid=cb.from.id,cid=cb.message.chat.id;if(!d.startsWith("sub:check:")&&!d.startsWith("set_lang:"))await this.tg.answerCallbackQuery(cb.id).catch(()=>{});if(d.startsWith("set_lang:")){const lang=d.split(":")[1];await this.db.setLang(uid,languages.includes(lang as any)?lang:"en");await this.tg.answerCallbackQuery(cb.id,{text:text("language_set",lang)}).catch(()=>{});await this.tg.deleteMessage(cid,cb.message.message_id).catch(()=>{});const fake:{message_id:number;chat:{id:number};from:any}={message_id:0,chat:{id:cid},from:cb.from};await this.start(fake);return}if(d==="sub:choose"){const lang=await this.lang(uid);await this.buttons(cid,text("payment_method",lang),kb([[{text:`⭐ Stars (${LIMITS.priceStars}⭐)`,callback_data:"sub:stars"}],[{text:`💎 Crypto (${LIMITS.priceCryptoUsd}$)`,callback_data:"sub:crypto"}]]));return}if(d==="sub:stars"){const lang=await this.lang(uid);await this.tg.sendInvoice(cid,`${text("subscription",lang)} Threads Bot`,`30 ${text("days",lang)}`,`sub_${uid}`,LIMITS.priceStars);return}if(d==="sub:crypto")return this.crypto(cid,uid);if(d.startsWith("sub:check:"))return this.cryptoCheck(cb,Number(d.split(":")[2]));if(d.startsWith("sup:"))return this.supportCallback(cb);if(d.startsWith("ticket:"))return this.ticketCallback(cb);if(d.startsWith("adm:"))return this.adminCallback(cb);if(d.startsWith("cmt:"))return this.commentsPage(cb);if(/^(text|img):[\w.]+:\d+$/.test(d))return this.choice(cb)}
 private async crypto(cid:number,uid:number){const lang=await this.lang(uid),response=await fetch("https://pay.crypt.bot/api/createInvoice",{method:"POST",headers:{"content-type":"application/json","Crypto-Pay-API-Token":this.env.CRYPTO_BOT_TOKEN},body:JSON.stringify({asset:"USDT",amount:String(LIMITS.priceCryptoUsd),description:"Threads Bot Subscription — 30 days",payload:String(uid),paid_btn_name:"callback",paid_btn_url:`https://t.me/${(await this.tg.getMe()).username}`})});const data:any=await response.json();if(!data.ok){await this.tg.sendMessage(cid,"❌ Error.");return}await this.buttons(cid,`💎 <b>${LIMITS.priceCryptoUsd} USDT</b>\n${text("after_payment",lang)} «${text("i_paid",lang)}».`,kb([[{text:`💎 ${text("pay",lang)}`,url:data.result.pay_url}],[{text:`✅ ${text("i_paid",lang)}`,callback_data:`sub:check:${data.result.invoice_id}`}]]))}
 private async cryptoCheck(cb:CallbackQuery,id:number){const response=await fetch(`https://pay.crypt.bot/api/getInvoices?invoice_ids=${id}`,{headers:{"Crypto-Pay-API-Token":this.env.CRYPTO_BOT_TOKEN}}),data:any=await response.json(),lang=await this.lang(cb.from.id);if(data.ok&&data.result.items?.[0]?.status==="paid"){const exp=await this.db.activate(cb.from.id,"crypto",LIMITS.priceCryptoUsd);await this.db.logEvent(cb.from.id,"subscribe","crypto");await this.tg.answerCallbackQuery(cb.id,{text:"✅!",show_alert:true}).catch(()=>{});await this.tg.editText(cb.message!.chat.id,cb.message!.message_id,`🎉 <b>${text("paid",lang)}!</b> ${text("until",lang)} ${fmtDate(exp)}`)}else await this.tg.answerCallbackQuery(cb.id,{text:`⏳ ${text("not_found",lang)}`,show_alert:true}).catch(()=>{})}
 private async supportCallback(cb:CallbackQuery){const d=cb.data!,uid=cb.from.id,cid=cb.message!.chat.id,lang=await this.lang(uid);if(d.startsWith("sup:write:")){const type=d.split(":")[2];await this.db.setState(uid,"waiting_support",type);await this.buttons(cid,`${type==="suggestion"?"💡":"❓"} <b>${text(type==="suggestion"?"suggestion":"question",lang)}</b>\n\nSend your message:`,kb([[{text:"❌ Cancel",callback_data:"sup:cancel"}]]));return}if(d==="sup:cancel"){await this.db.clearState(uid,"waiting_support");await this.tg.editText(cid,cb.message!.message_id,"❌ Cancelled.");return}const tickets=await this.db.tickets(uid);let value=tickets.length?`📋 <b>${text("my_tickets",lang)}:</b>\n\n`: `📋 ${text("my_tickets",lang)}: —`;for(const t of tickets)value+=`${t.status==="answered"?"✅":"⏳"}${t.ticket_type==="suggestion"?"💡":"❓"} <b>#${t.id}</b> (${String(t.created_at).slice(0,10)})\n   ${esc(String(t.message).slice(0,80))}\n${t.answer?`   💬 ${esc(String(t.answer).slice(0,80))}\n`:""}\n`;await this.buttons(cid,value,kb([[{text:`❓ ${text("question",lang)}`,callback_data:"sup:write:question"},{text:`💡 ${text("suggestion",lang)}`,callback_data:"sup:write:suggestion"}]]))}
 private async ticketCallback(cb:CallbackQuery){if(!isAdmin(this.env,cb.from.id))return;if(cb.data==="ticket:cancel"){await this.db.clearState(cb.from.id,"admin_reply");await this.tg.editText(cb.message!.chat.id,cb.message!.message_id,"❌ Cancelled.");return}const id=Number(cb.data!.split(":")[2]);await this.db.setState(cb.from.id,"admin_reply",String(id));await this.buttons(cb.message!.chat.id,`💬 Reply to <b>#${id}</b>:`,kb([[{text:"❌ Cancel",callback_data:"ticket:cancel"}]]))}
 private async adminCallback(cb:CallbackQuery){if(!isAdmin(this.env,cb.from.id))return;const action=cb.data!.split(":")[1],cid=cb.message!.chat.id,back=kb([[{text:"◀️",callback_data:"adm:back"}]]);if(action==="back"){const counts=await this.db.accountCounts(),subs=await this.db.subscribers(),tickets=await this.db.tickets();await this.buttons(cid,`🔐 🟢${counts.alive||0}/${counts.total}|👥${subs.length}|🆘${tickets.length}`,this.adminKb(tickets.length));return}if(action==="stats"||action==="health"||action==="detailed"){const stats:any[]=await this.db.accountStats();let value=`${action==="stats"?"📊 <b>Stats</b>":action==="health"?"🏥 <b>Health</b>":"📋 <b>Details</b>"}\n\n`;for(const s of stats)value+=`${s.is_alive?"🟢":"🔴"} <b>${esc(s.name)}</b>\n   📨${s.requests_count}|⏰${s.hourly_requests}/${LIMITS.accountHourly}|📝${s.posts_sent}|❌${s.errors_count}\n   🕐${s.last_used?String(s.last_used).slice(11,19):"—"}|${esc(s.last_error||"—")}\n\n`;await this.buttons(cid,value,back);return}if(action==="reload"){const result=await reviveAccounts(this.env);await this.buttons(cid,result.length?`🔄\n${result.map(x=>`${x.ok?"🟢":"🔴"} ${esc(x.name)}`).join("\n")}`:"✅ All ok!",back);return}if(action==="banlist"){const rows:any[]=await this.db.banned();await this.buttons(cid,rows.length?`🚫 (${rows.length})\n\n${rows.map(r=>`• <code>${r.user_id}</code> — ${esc(r.reason)}`).join('\n')}`:"🚫 Empty",back);return}if(action==="subs"){const rows:any[]=await this.db.subscribers();await this.buttons(cid,rows.length?`👥 (${rows.length})\n\n${rows.map(r=>`• <code>${r.user_id}</code> ${Math.max(0,Math.floor((new Date(r.expires_at).getTime()-Date.now())/86_400_000))}d|${esc(r.payment_method)}`).join('\n')}`:"👥 None",back);return}if(action==="tickets"){const rows:any[]=await this.db.tickets();let value=rows.length?`🆘 Open (${rows.length}):\n\n`:`🆘 No open tickets.`;for(const r of rows.slice(0,10))value+=`${r.ticket_type==="suggestion"?"💡":"❓"} <b>#${r.id}</b> @${esc(r.username||r.user_id)} (${String(r.created_at).slice(0,10)})\n   ${esc(String(r.message).slice(0,80))}\n\n`;if(rows.length)value+="\n💬 <b>Quick reply:</b> <code>ID text</code>";await this.buttons(cid,value,back);return}if(action==="analytics"){const a=await this.db.analytics();await this.buttons(cid,`📈 <b>Analytics (24h)</b>\n\n👥 New users: ${a.newUsers}\n🔥 DAU: ${a.dau}\n📊 Active (7d): ${a.active7d}\n\n📨 Total requests: ${a.requests}\n📝 Text: ${a.text} | 📸 Img: ${a.img} | 💬 Cmts: ${a.comments}\n\n🚫 Hit limits: ${a.exhausted}\n💰 New subs: ${a.newSubs}\n💵 Revenue: $${a.revenue.toFixed(2)}`,back)}}
 private async choice(cb:CallbackQuery){const [mode,username,pageRaw]=cb.data!.split(":"),page=Number(pageRaw),uid=cb.from.id,cid=cb.message!.chat.id,lang=await this.lang(uid);if(page===0){const access=await this.access(uid);if(!access.ok){await this.buttons(cid,"🔒",kb([[{text:text("subscribe_btn",lang),callback_data:"sub:choose"}]]));return}const limited=await this.db.rateLimit(uid);if(limited){await this.tg.editText(cid,cb.message!.message_id,`⏳ ${limited}`);return}}await this.tg.editMarkup(cid,cb.message!.message_id).catch(()=>{});const loading=await this.tg.sendMessage(cid,`⏳ @${esc(username)}...`);try{let posts=mode==="text"?await this.db.cache<Post[]>(username,mode):null,status:"ok"|string="ok";if(!posts){if(page===0)await Promise.all([this.db.logRequest(uid,username),this.db.logEvent(uid,"search",username),this.db.logEvent(uid,"request",`${mode}:${username}`)]);const fetched=await fetchPosts(this.env,username,mode as "text"|"img",20);posts=fetched.data;status=fetched.status;if(posts&&status==="ok"&&mode==="text")await this.db.setCache(username,mode,posts);if(posts&&status==="ok"&&page===0&&!await this.db.hasSubscription(uid)&&!isAdmin(this.env,uid))await this.db.logEvent(uid,"free_request",username)}await this.tg.deleteMessage(cid,loading.message_id).catch(()=>{});if(status==="all_dead"){await this.alert(`Все аккаунты недоступны! @${username}`);await this.tg.sendMessage(cid,text("all_dead",lang));return}if(status==="user_not_found"){await this.tg.sendMessage(cid,text("user_not_found",lang,{username}));return}if(!posts?.length){await this.tg.sendMessage(cid,text("no_posts",lang));return}await this.db.setState(cid,"last_username",username);const start=page*5,current=posts.slice(start,start+5),end=start+current.length;if(!current.length){await this.tg.sendMessage(cid,text("no_more",lang));return}if(page===0)await this.tg.sendMessage(cid,text("posts_found",lang,{username,count:posts.length}));for(let i=0;i<current.length;i++){const p=current[i];if(mode==="text"){let indicator=p.has_image&&p.has_video?` (📷🎥 ${text("photo",lang)}+${text("video",lang)})`:p.has_image?` (📷 ${text("photo",lang)})`:p.has_video?` (🎥 ${text("video",lang)})`:"";await this.tg.sendMessage(cid,`<b>${start+i+1}.</b> ${esc(p.text.slice(0,4000))}${indicator}`)}else if(p.image)await this.tg.sendPhoto(cid,p.image,`${start+i+1}.`)}const more=posts.length>end,rows:any[][]=[];if(more)rows.push([{text:`➡️ ${text("more",lang)} (${end+1}–${Math.min(end+5,posts.length)})`,callback_data:`${mode}:${username}:${page+1}`}]);rows.push([{text:mode==="text"?text("screens_btn",lang):text("text_btn",lang),callback_data:`${mode==="text"?"img":"text"}:${username}:0`}]);await this.buttons(cid,`✅ <b>${start+1}–${end} ${text("of",lang)} ${posts.length}</b>\n${more?text("more_available",lang):`📭 ${text("all",lang)}`}\n${text("reply_for_comments",lang)}\n${text("send_other",lang)}`,kb(rows))}catch(error){await this.tg.deleteMessage(cid,loading.message_id).catch(()=>{});await this.alert(`Bot error: ${error}`);await this.tg.sendMessage(cid,`❌ <code>${esc(String(error).slice(0,300))}</code>`)}}
 private async replyComments(m:TgMessage){const uid=m.from!.id,cid=m.chat.id,lang=await this.lang(uid),replied=m.reply_to_message!,match=(replied.text||replied.caption||"").match(/^(?:📷|🎥)?\s*(?:<b>)?(\d+)\./),fallback=(m.text||"").match(/^(\d+)$/),number=Number(match?.[1]||fallback?.[1]);if(!number)return;const username=await this.db.state(cid,"last_username");if(!username){await this.tg.sendMessage(cid,"❌ Send a username first.");return}const access=await this.access(uid);if(!access.ok){await this.db.logEvent(uid,"free_exhausted",access.kind);await this.buttons(cid,access.note,kb([[{text:text("subscribe_btn",lang),callback_data:"sub:choose"}]]));return}const limited=await this.db.rateLimit(uid);if(limited){await this.tg.sendMessage(cid,`⏳ ${limited}`);return}const loading=await this.tg.sendMessage(cid,"💬 Загружаю комментарии...");try{const found=await fetchComments(this.env,username,number-1,20);await this.tg.deleteMessage(cid,loading.message_id).catch(()=>{});if(found.status==="all_dead"){await this.tg.sendMessage(cid,text("all_dead",lang));return}if(found.status==="post_not_found"||found.status==="user_not_found"){await this.tg.sendMessage(cid,"❌ Post not found.");return}if(!found.data?.length){await this.tg.sendMessage(cid,"😔 No comments.");return}await Promise.all([this.db.logRequest(uid,`${username}/cmt/${number}`),this.db.logEvent(uid,"comments_request",username),this.db.logEvent(uid,"request",`comments:${username}`)]);if(!await this.db.hasSubscription(uid)&&!isAdmin(this.env,uid))await this.db.logEvent(uid,"free_request",`cmt:${username}/${number}`);const key=`${username}_cmt_${number-1}`;await this.db.setCache(key,"comments",found.data.map(c=>({text:c.text,author:c.author})));await this.tg.sendMessage(cid,`💬 <b>@${esc(username)}</b> (${found.data.length})`);await this.sendComments(cid,found.data.slice(0,5),0);if(found.data.length>5)await this.buttons(cid,`✅ 1–5 ${text("of",lang)} ${found.data.length}`,kb([[{text:`➡️ ${text("more",lang)} (6–${Math.min(10,found.data.length)})`,callback_data:`cmt:${username}:${number-1}:1`}]]));else await this.tg.sendMessage(cid,`✅ ${text("all",lang)} ${found.data.length}.`)}catch(error){await this.tg.deleteMessage(cid,loading.message_id).catch(()=>{});await this.alert(`Bot error: ${error}`);await this.tg.sendMessage(cid,`❌ <code>${esc(String(error).slice(0,300))}</code>`)}}
 private async sendComments(cid:number,comments:Comment[],start:number){for(let i=0;i<comments.length;i++)await this.tg.sendMessage(cid,`<b>${start+i+1}. ${esc(comments[i].author||"—")}</b>\n${esc(comments[i].text.slice(0,1000))}`)}
 private async commentsPage(cb:CallbackQuery){const [,username,indexRaw,pageRaw]=cb.data!.split(":"),index=Number(indexRaw),page=Number(pageRaw),cid=cb.message!.chat.id,lang=await this.lang(cb.from.id),comments=await this.db.cache<Comment[]>(`${username}_cmt_${index}`,"comments");await this.tg.editMarkup(cid,cb.message!.message_id).catch(()=>{});if(!comments){await this.tg.sendMessage(cid,"⏳ Cache expired.");return}const start=page*5,current=comments.slice(start,start+5),end=start+current.length;if(!current.length){await this.tg.sendMessage(cid,text("no_more",lang));return}await this.sendComments(cid,current,start);if(comments.length>end)await this.buttons(cid,`✅ ${start+1}–${end} ${text("of",lang)} ${comments.length}`,kb([[{text:`➡️ ${text("more",lang)} (${end+1}–${Math.min(end+5,comments.length)})`,callback_data:`cmt:${username}:${index}:${page+1}`}]]));else await this.tg.sendMessage(cid,`✅ ${text("all",lang)} ${comments.length}.`)}
}


### `test/i18n.test.ts`


In [ ]:
%%writefile /content/threadsbot-cloudflare/test/i18n.test.ts
import { describe, expect, it } from "vitest";
import { cleanPostText, text } from "../src/i18n";

describe("i18n",()=>{
 it("formats placeholders",()=>expect(text("posts_found","en",{username:"zuck",count:3})).toContain("@zuck"));
 it("falls back for unknown language",()=>expect(text("no_posts","xx")).toBe(text("no_posts","ru")));
 it("removes Threads translation suffix",()=>expect(cleanPostText("Hello world\nSee translation")).toBe("Hello world"));
});


## 3. Установка и проверка проекта


In [ ]:
os.chdir(PROJECT)
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npm", "run", "typecheck"], check=True)
subprocess.run(["npm", "test"], check=True)
print("✅ Код собирается, тесты пройдены")


## 4. GitHub API: новый репозиторий и push

Создайте GitHub Personal Access Token. Для classic token нужна область `repo`; для fine-grained token — доступ к репозиториям и право **Contents: Read and write**. Ввод скрыт. По умолчанию создаётся публичный `threadsbot-cloudflare`.

Токен используется только из памяти. В `origin` сохраняется обычный URL без токена.


In [ ]:
github_token = getpass.getpass("GitHub Personal Access Token: ").strip()
if not github_token:
    raise ValueError("GitHub token обязателен")
headers = {"Authorization": f"Bearer {github_token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
me = requests.get("https://api.github.com/user", headers=headers, timeout=30)
me.raise_for_status()
login = me.json()["login"]
owner = input(f"Владелец репозитория [{login}]: ").strip() or login
repo_name = input("Имя репозитория [threadsbot-cloudflare]: ").strip() or "threadsbot-cloudflare"
private = (input("Приватный репозиторий? [y/N]: ").strip().lower() == "y")

check = requests.get(f"https://api.github.com/repos/{owner}/{repo_name}", headers=headers, timeout=30)
if check.status_code == 404:
    endpoint = "https://api.github.com/user/repos" if owner.lower() == login.lower() else f"https://api.github.com/orgs/{owner}/repos"
    created = requests.post(endpoint, headers=headers, json={"name": repo_name, "private": private, "description": "Threads Reader Bot for Cloudflare Workers"}, timeout=30)
    created.raise_for_status()
elif not check.ok:
    check.raise_for_status()

# Кладём копию установочного Colab в новый репозиторий, если исходная ветка доступна.
notebook_url = "https://raw.githubusercontent.com/Bergaff/threadsbot/arena/01a0055c-threadsbot/notebooks/threadsbot_cloudflare_deploy.ipynb"
notebook_response = requests.get(notebook_url, timeout=60)
if notebook_response.ok:
    (PROJECT / "notebooks" / "threadsbot_cloudflare_deploy.ipynb").write_bytes(notebook_response.content)

subprocess.run(["git", "init", "-b", "main"], cwd=PROJECT, check=True)
subprocess.run(["git", "config", "user.name", login], cwd=PROJECT, check=True)
subprocess.run(["git", "config", "user.email", f"{login}@users.noreply.github.com"], cwd=PROJECT, check=True)
subprocess.run(["git", "add", "."], cwd=PROJECT, check=True)
subprocess.run(["git", "commit", "-m", "Deploy Threads bot on Cloudflare Workers"], cwd=PROJECT, check=True)
remote = f"https://github.com/{owner}/{repo_name}.git"
subprocess.run(["git", "remote", "add", "origin", remote], cwd=PROJECT, check=True)

# Askpass не содержит токен: он читает его из переменной окружения процесса.
askpass = Path("/content/github-askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "x-access-token";; *) echo "$GITHUB_TOKEN";; esac\n', "utf-8")
askpass.chmod(0o700)
push_env = os.environ.copy()
push_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    subprocess.run(["git", "push", "-u", "origin", "main"], cwd=PROJECT, env=push_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    push_env.pop("GITHUB_TOKEN", None)
print(f"✅ Код отправлен: https://github.com/{owner}/{repo_name}")


## 5. Ввод Cloudflare API и получение D1 UUID

Нужны Cloudflare API Token с правами редактирования **Workers Scripts, D1, Queues и Browser Rendering**, а также Account ID. Ячейка создаёт или находит базу `threadsbot`, выводит её UUID (это идентификатор, не секрет), создаёт Queue и применяет миграции.


In [ ]:
cloudflare_token = getpass.getpass("Cloudflare API Token: ").strip()
cloudflare_account_id = getpass.getpass("Cloudflare Account ID: ").strip()
if not cloudflare_token or not cloudflare_account_id:
    raise ValueError("Cloudflare Token и Account ID обязательны")
os.environ["CLOUDFLARE_API_TOKEN"] = cloudflare_token
os.environ["CLOUDFLARE_ACCOUNT_ID"] = cloudflare_account_id
subprocess.run(["npx", "wrangler", "whoami"], cwd=PROJECT, check=True)
print("✅ Cloudflare API подключён")


In [ ]:
def run(*args, capture=False, input_value=None):
    return subprocess.run(list(args), cwd=PROJECT, check=True, text=True,
                          capture_output=capture, input=input_value)

listed = run("npx", "wrangler", "d1", "list", "--json", capture=True)
databases = json.loads(listed.stdout)
db = next((item for item in databases if item.get("name") == "threadsbot"), None)
if db is None:
    created = run("npx", "wrangler", "d1", "create", "threadsbot", "--location", "eeur", capture=True)
    output = created.stdout + created.stderr
    match = re.search(r'database_id\s*=\s*"([0-9a-f-]{36})"', output)
    if not match:
        raise RuntimeError("Не удалось получить D1 UUID:\n" + output)
    d1_database_id = match.group(1)
else:
    d1_database_id = db.get("uuid") or db.get("id")
if not d1_database_id:
    raise RuntimeError("D1 UUID отсутствует в ответе Cloudflare")

config_path = PROJECT / "wrangler.toml"
config = config_path.read_text("utf-8")
config, replacements = re.subn(r'database_id\s*=\s*"[^"]+"', f'database_id = "{d1_database_id}"', config, count=1)
if replacements != 1:
    raise RuntimeError("database_id не найден в wrangler.toml")
config_path.write_text(config, "utf-8")

queues = run("npx", "wrangler", "queues", "list", capture=True)
if "threadsbot-updates" not in queues.stdout:
    run("npx", "wrangler", "queues", "create", "threadsbot-updates")
run("npm", "run", "db:remote")
print("✅ D1 UUID:", d1_database_id)
print("✅ Queue: threadsbot-updates")


## 6. Приватная загрузка cookies

Выберите JSON-файлы Cookie-Editor/Playwright. Они не добавляются в Git: репозиторий уже запушен, `accounts/` находится в `.gitignore`, а файлы создаются вне проекта. После импорта локальные копии удаляются даже при ошибке.


In [ ]:
from google.colab import files
PRIVATE_ACCOUNTS = Path("/content/private_threads_accounts")
shutil.rmtree(PRIVATE_ACCOUNTS, ignore_errors=True)
PRIVATE_ACCOUNTS.mkdir(mode=0o700)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Cookies не выбраны")
for filename, content in uploaded.items():
    safe = Path(filename).name
    if not safe.endswith(".json"):
        raise ValueError(f"Разрешены только .json: {safe}")
    parsed = json.loads(content)
    if not isinstance(parsed, list) or not parsed:
        raise ValueError(f"Ожидался непустой список cookies: {safe}")
    target = PRIVATE_ACCOUNTS / safe
    target.write_bytes(content); target.chmod(0o600)
try:
    run("npm", "run", "accounts:import", "--", str(PRIVATE_ACCOUNTS), "--remote")
finally:
    shutil.rmtree(PRIVATE_ACCOUNTS, ignore_errors=True)
    uploaded.clear()
print("✅ Cookies импортированы в D1 и удалены из Colab")


## 7. Необязательный перенос `bot.db`

Пропустите ячейку, если старые пользователи, подписки, тикеты и аналитика не нужны.


In [ ]:
legacy = files.upload()
if legacy:
    filename, content = next(iter(legacy.items()))
    if Path(filename).name != "bot.db":
        raise ValueError("Нужно выбрать bot.db")
    private_db = Path("/content/private_bot.db")
    private_db.write_bytes(content); private_db.chmod(0o600)
    try:
        run("python3", "scripts/migrate_legacy.py", "--db", str(private_db), "--accounts", "/content/no-accounts")
    finally:
        private_db.unlink(missing_ok=True)
        legacy.clear()
    print("✅ SQLite перенесён в D1; временный файл удалён")


## 8. Секреты Worker, deploy и Telegram webhook


In [ ]:
# Создаём Worker до загрузки secrets: webhook ещё не установлен, поэтому он не принимает обновления.
subprocess.run(["npx", "wrangler", "deploy"], cwd=PROJECT, check=True, stdout=subprocess.DEVNULL)

telegram_token = getpass.getpass("TELEGRAM_TOKEN: ").strip()
crypto_token = getpass.getpass("CRYPTO_BOT_TOKEN: ").strip()
webhook_secret = getpass.getpass("WEBHOOK_SECRET (Enter = сгенерировать): ").strip() or secrets.token_urlsafe(32)
if not telegram_token or not crypto_token:
    raise ValueError("Токены Telegram и Crypto Bot обязательны")

def put_secret(name, value):
    subprocess.run(["npx", "wrangler", "secret", "put", name], cwd=PROJECT,
                   input=value + "\n", text=True, check=True,
                   stdout=subprocess.DEVNULL)
for name, value in (("TELEGRAM_TOKEN", telegram_token), ("CRYPTO_BOT_TOKEN", crypto_token), ("WEBHOOK_SECRET", webhook_secret)):
    put_secret(name, value)
print("✅ Worker secrets сохранены")


In [ ]:
deployed = run("npx", "wrangler", "deploy", capture=True)
output = deployed.stdout + deployed.stderr
print(output)
match = re.search(r'https://[^\s]+\.workers\.dev', output)
worker_url = match.group(0).rstrip("/") if match else input("URL Worker из вывода: ").strip().rstrip("/")
setup = requests.post(worker_url + "/setup-webhook", headers={"Authorization": "Bearer " + webhook_secret}, timeout=60)
setup.raise_for_status()
health = requests.get(worker_url + "/health", timeout=30)
health.raise_for_status()
print("Webhook:", setup.json())
print("Health:", health.json())
print("✅ Бот полностью развёрнут:", worker_url)


## 9. Обновление `wrangler.toml` в GitHub

D1 UUID появился после первого push. Эта ячейка делает второй безопасный коммит без секретов и отправляет актуальный `wrangler.toml`.


In [ ]:
subprocess.run(["git", "add", "wrangler.toml"], cwd=PROJECT, check=True)
changed = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=PROJECT).returncode != 0
if changed:
    subprocess.run(["git", "commit", "-m", "Configure Cloudflare D1 database"], cwd=PROJECT, check=True)
    askpass = Path("/content/github-askpass.sh")
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "x-access-token";; *) echo "$GITHUB_TOKEN";; esac\n', "utf-8")
    askpass.chmod(0o700)
    push_env = os.environ.copy(); push_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        subprocess.run(["git", "push", "origin", "main"], cwd=PROJECT, env=push_env, check=True)
    finally:
        askpass.unlink(missing_ok=True)
print("✅ GitHub содержит актуальный D1 UUID; токены и cookies не коммитились")


## 10. Обязательная очистка

После выполнения также выберите **Runtime → Disconnect and delete runtime**.


In [ ]:
for key in ("CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID", "GITHUB_TOKEN"):
    os.environ.pop(key, None)
for variable in ("github_token", "cloudflare_token", "cloudflare_account_id", "telegram_token", "crypto_token", "webhook_secret", "uploaded", "legacy"):
    globals().pop(variable, None)
shutil.rmtree("/content/private_threads_accounts", ignore_errors=True)
Path("/content/private_bot.db").unlink(missing_ok=True)
Path("/content/github-askpass.sh").unlink(missing_ok=True)
gc.collect()
print("✅ Секреты и временные приватные файлы удалены")
